In [7]:
import tensorflow as tf
import os
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers
import json

"""
PHYSICS-CONSTRAINED OPEN-SET LIVESTOCK IDENTIFICATION
FINAL CORRECTED VERSION

Data Strategy:
- face_train (345 cows): Training set - learn discriminative embeddings
- face_val (30 cows): Synthetic unknown source during curriculum training
- face_test: COMPLETELY UNTOUCHED - used only for final open-set evaluation

Training Protocol:
- Phase 1: Train on face_train only (100% known, physics warmup)
- Phase 2-3: Mix face_train (known) + face_val (as unknowns via heavy augmentation)
- Validation: Monitor on face_train subset (not face_val)
- Evaluation: face_test as true unknowns

Key Corrections:
1. UnknownRejectionLoss freezes class_weights (no gradient leak)
2. CenterLoss updates only during training (no data leakage)
3. Physics losses apply only to known samples (cleaner interpretation)
4. Proper data split: train on face_train, val serves as unknown source
"""

# Set seeds
np.random.seed(42)
tf.random.set_seed(42)

# GPU setup
gpus = tf.config.experimental.list_physical_devices('GPU')
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except:
        pass

dataset_root = "/home/student/Desktop/Kumar/dataset"

# ============================================================================
# CONFIGURATION
# ============================================================================

class CFG:
    IMG_SIZE = 224
    EMBED_DIM = 512
    MARGIN = 0.5
    SCALE = 64
    BATCH_SIZE = 32
    
    # Loss weights
    CENTER_LOSS_WEIGHT = 0.0001
    UNKNOWN_LOSS_WEIGHT = 0.1
    
    # Physics loss weights (final values after warmup)
    PHYSICS_SYM_WEIGHT = 0.01
    PHYSICS_ENERGY_WEIGHT = 0.005
    PHYSICS_LAP_WEIGHT = 0.01
    
    # Physics warmup
    PHYSICS_WARMUP_EPOCHS = 10
    
    # Energy temperature
    ENERGY_TAU = 0.1
    
    STEPS_PER_EPOCH = 150
    AUTOTUNE = tf.data.AUTOTUNE
    
    # Curriculum schedule
    PHASE_1_EPOCHS = 5   # Warmup: 100% known from face_train
    PHASE_2_EPOCHS = 15  # Introduction: 80% known, 20% unknown (from face_val)
    PHASE_3_EPOCHS = 10  # Hardening: 60% known, 40% unknown (from face_val)
    
    TOTAL_EPOCHS = PHASE_1_EPOCHS + PHASE_2_EPOCHS + PHASE_3_EPOCHS

print("=" * 70)
print("PHYSICS-CONSTRAINED OPEN-SET LIVESTOCK IDENTIFICATION")
print("FINAL CORRECTED VERSION")
print("=" * 70)
print(f"\nData Strategy:")
print(f"  face_train: Training set (learn identities)")
print(f"  face_val:   Unknown source (synthetic unknowns during training)")
print(f"  face_test:  UNTOUCHED (final evaluation only)")

# ============================================================================
# DATA LOADING
# ============================================================================

def load_image(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, (CFG.IMG_SIZE, CFG.IMG_SIZE))
    img = tf.cast(img, tf.float32)
    img = tf.keras.applications.efficientnet.preprocess_input(img)
    return img

def augment_image(img):
    """Standard augmentation for known samples"""
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, 0.25)
    img = tf.image.random_contrast(img, 0.8, 1.2)
    img = tf.image.random_saturation(img, 0.8, 1.2)
    return img

def light_augmentation(images):
    """Light augmentation for L_lap (batch-safe)"""
    images = tf.image.random_brightness(images, 0.15)
    images = tf.image.random_contrast(images, 0.9, 1.1)
    return images

def augment_unknown(img):
    """Heavy augmentation to create synthetic unknowns from face_val"""
    img = tf.image.random_brightness(img, 0.5)
    img = tf.image.random_contrast(img, 0.5, 1.5)
    img = tf.image.random_saturation(img, 0.5, 1.5)
    img = tf.image.random_hue(img, 0.2)
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    img = tf.image.rot90(img, k=tf.random.uniform([], 0, 4, dtype=tf.int32))
    noise = tf.random.normal(shape=tf.shape(img), mean=0.0, stddev=0.1)
    img = img + noise
    img = tf.clip_by_value(img, -2.0, 2.0)
    return img

class DatasetBuilder:
    """Load cow images from a single folder"""
    def __init__(self, folder_path, seed=42):
        self.cow_data = {}
        np.random.seed(seed)
        
        if not os.path.exists(folder_path):
            raise ValueError(f"Folder not found: {folder_path}")
        
        for cow_id in sorted(os.listdir(folder_path)):
            cow_path = os.path.join(folder_path, cow_id)
            if not os.path.isdir(cow_path):
                continue
            
            imgs = [os.path.join(cow_path, f) for f in os.listdir(cow_path)
                   if f.lower().endswith((".jpg", ".jpeg", ".png"))]
            
            if len(imgs) > 0:
                self.cow_data[cow_id] = imgs
        
        self.classes = sorted(list(self.cow_data.keys()))
        self.cls_to_idx = {c: i for i, c in enumerate(self.classes)}
        self.num_classes = len(self.classes)
        self.total_images = sum(len(v) for v in self.cow_data.values())
    
    def get_tfdata_unbatched(self, shuffle=True, augment=False):
        """Get unbatched dataset for this folder"""
        face_paths = []
        labels = []
        
        for cow_id, face_list in self.cow_data.items():
            for face_path in face_list:
                face_paths.append(face_path)
                labels.append(self.cls_to_idx[cow_id])
        
        ds = tf.data.Dataset.from_tensor_slices({
            'face_path': face_paths,
            'label': labels
        })
        
        def map_fn(sample):
            face_img = load_image(sample['face_path'])
            
            if augment:
                face_img = augment_image(face_img)
            
            return {
                'face_input': face_img,
                'label_input': sample['label']
            }, sample['label']
        
        ds = ds.map(map_fn, num_parallel_calls=tf.data.AUTOTUNE)
        
        if shuffle:
            ds = ds.shuffle(2000, seed=42, reshuffle_each_iteration=True)
        
        ds = ds.repeat()
        return ds

# Load data
print("\n" + "=" * 70)
print("LOADING DATA")
print("=" * 70)

face_train_path = os.path.join(dataset_root, "face_train")
face_val_path = os.path.join(dataset_root, "face_val")

train_data = DatasetBuilder(face_train_path)
unknown_source_data = DatasetBuilder(face_val_path)  # Will be used as unknowns

print(f"\n✓ Training set (face_train):")
print(f"  {train_data.num_classes} cows, {train_data.total_images} images")
print(f"  Classes: {train_data.classes[:5]}...")

print(f"\n✓ Unknown source (face_val):")
print(f"  {unknown_source_data.num_classes} cows, {unknown_source_data.total_images} images")
print(f"  These cows will serve as synthetic unknowns during training")
print(f"  Classes: {unknown_source_data.classes[:5]}...")

print(f"\n✓ Test set (face_test): UNTOUCHED")
print(f"  Will be used only for final open-set evaluation")

# ============================================================================
# MODEL ARCHITECTURE
# ============================================================================

def build_model(num_classes, embedding_size=512, margin=0.5, scale=64):
    """Build model with named embedding layer"""
    face_input = keras.Input(shape=(224, 224, 3), name="face_input")
    label_input = keras.Input(shape=(), dtype=tf.int32, name="label_input")
    
    backbone = keras.applications.EfficientNetB0(
        include_top=False,
        input_shape=(224, 224, 3),
        weights="imagenet"
    )
    
    for layer in backbone.layers[:-30]:
        layer.trainable = False
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False
    
    x = backbone(face_input, training=True)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.4)(x)
    
    x = layers.Dense(embedding_size, kernel_regularizer=keras.regularizers.l2(1e-4),
                     name="embedding_dense")(x)
    
    embeddings = layers.Lambda(
        lambda x: tf.math.l2_normalize(x, axis=1),
        name="l2_normalize"
    )(x)
    
    class SafeArcFaceLayer(layers.Layer):
        def __init__(self, n_classes, margin=0.5, scale=64, **kwargs):
            super().__init__(**kwargs)
            self.n_classes = n_classes
            self.margin = margin
            self.scale = scale
            self.cos_m = tf.cos(margin)
            self.sin_m = tf.sin(margin)
        
        def build(self, input_shape):
            self.W = self.add_weight(
                name="arcface_weights",
                shape=(input_shape[0][-1], self.n_classes),
                initializer=keras.initializers.HeNormal(),
                trainable=True,
                regularizer=keras.regularizers.l2(1e-4)
            )
            super().build(input_shape)
        
        def call(self, inputs):
            embeddings, labels = inputs
            
            W_norm = tf.nn.l2_normalize(self.W, axis=0)
            embeddings_norm = tf.nn.l2_normalize(embeddings, axis=1)
            
            cos_theta = tf.matmul(embeddings_norm, W_norm)
            cos_theta = tf.clip_by_value(cos_theta, -1.0 + 1e-7, 1.0 - 1e-7)
            
            sin_theta = tf.sqrt(tf.maximum(1.0 - tf.square(cos_theta), 1e-7))
            target_logits = cos_theta * self.cos_m - sin_theta * self.sin_m
            
            labels = tf.cast(labels, tf.int32)
            known_mask = tf.greater_equal(labels, 0)
            labels_safe = tf.where(known_mask, labels, tf.zeros_like(labels))
            
            onehot_raw = tf.one_hot(labels_safe, depth=self.n_classes, dtype=tf.float32)
            onehot = onehot_raw * tf.expand_dims(tf.cast(known_mask, tf.float32), axis=1)
            
            output = cos_theta * (1.0 - onehot) + target_logits * onehot
            output = output * self.scale
            
            return output
        
        def get_config(self):
            return {"n_classes": self.n_classes, "margin": self.margin, "scale": self.scale}
    
    logits = SafeArcFaceLayer(num_classes, margin=margin, scale=scale,
                              name="arcface")([embeddings, label_input])
    
    model = keras.Model(
        inputs=[face_input, label_input],
        outputs=[logits, embeddings],
        name="physics_constrained_model"
    )
    
    return model

print("\n" + "=" * 70)
print("BUILDING MODEL")
print("=" * 70)

# CRITICAL: Model has num_classes = face_train only (345 cows)
model = build_model(
    train_data.num_classes,  # Only train_data classes!
    embedding_size=CFG.EMBED_DIM,
    margin=CFG.MARGIN,
    scale=CFG.SCALE
)

print(f"\n✓ Model built for {train_data.num_classes} training classes")
print(f"  Architecture: EfficientNetB0 → Dense(512) → L2-Normalize → ArcFace")

# Embedding-only model
embedding_model = keras.Model(
    inputs=model.input[0],
    outputs=model.get_layer("l2_normalize").output,
    name="embedding_extractor"
)

print(f"✓ Embedding extraction model created")

# ============================================================================
# LOSSES (with all corrections)
# ============================================================================

class CenterLossLayer(layers.Layer):
    """CORRECTED: Only update centers during training"""
    def __init__(self, num_classes, embedding_dim, alpha=0.5, **kwargs):
        super().__init__(**kwargs)
        self.num_classes = num_classes
        self.embedding_dim = embedding_dim
        self.alpha = tf.constant(alpha, dtype=tf.float32)
    
    def build(self, input_shape):
        self.centers = self.add_weight(
            name="class_centers",
            shape=(self.num_classes, self.embedding_dim),
            initializer="zeros",
            trainable=False
        )
        super().build(input_shape)
    
    def call(self, embeddings, labels, training=True):
        labels = tf.cast(labels, tf.int32)
        known_mask = labels >= 0
        known_embeddings = tf.boolean_mask(embeddings, known_mask)
        known_labels = tf.boolean_mask(labels, known_mask)
        
        if tf.shape(known_embeddings)[0] == 0:
            return tf.constant(0.0, dtype=tf.float32)
        
        centers_batch = tf.gather(self.centers, known_labels)
        diff = known_embeddings - centers_batch
        loss = tf.reduce_mean(tf.reduce_sum(tf.square(diff), axis=1))
        
        if not training:
            return loss
        
        # Update centers (only during training)
        unique_labels, idx = tf.unique(known_labels)
        num_unique = tf.shape(unique_labels)[0]
        summed = tf.math.unsorted_segment_sum(known_embeddings, idx, num_unique)
        counts = tf.math.unsorted_segment_sum(tf.ones_like(idx, dtype=tf.float32), idx, num_unique)
        counts = tf.reshape(counts, (-1, 1))
        new_centers = summed / (counts + 1e-8)
        
        current_centers = tf.gather(self.centers, unique_labels)
        updated = self.alpha * current_centers + (1.0 - self.alpha) * new_centers
        
        scatter_idx = tf.expand_dims(unique_labels, axis=1)
        self.centers.assign(tf.tensor_scatter_nd_update(self.centers, scatter_idx, updated))
        
        return loss

class UnknownRejectionLoss(layers.Layer):
    """CORRECTED: Freeze class_weights"""
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
    
    def call(self, embeddings, labels, class_weights):
        labels = tf.cast(labels, tf.int32)
        unknown_mask = tf.less(labels, 0)
        unknown_embeddings = tf.boolean_mask(embeddings, unknown_mask)
        
        if tf.shape(unknown_embeddings)[0] == 0:
            return tf.constant(0.0, dtype=tf.float32)
        
        W_frozen = tf.stop_gradient(class_weights)  # CORRECTED
        W_norm = tf.nn.l2_normalize(W_frozen, axis=0)
        
        similarities = tf.matmul(unknown_embeddings, W_norm)
        max_similarities = tf.reduce_max(similarities, axis=1)
        loss = tf.reduce_mean(max_similarities)
        
        return loss

center_loss_layer = CenterLossLayer(train_data.num_classes, CFG.EMBED_DIM, alpha=0.5)
unknown_rejection_layer = UnknownRejectionLoss()

# ============================================================================
# PHYSICS LOSSES
# ============================================================================

def get_physics_weights_tf(epoch, warmup_epochs=10):
    """TF-native weight scheduler"""
    epoch = tf.cast(epoch, tf.float32)
    warmup_epochs = tf.cast(warmup_epochs, tf.float32)
    scale = tf.minimum(epoch / warmup_epochs, 1.0)
    
    return (
        CFG.PHYSICS_SYM_WEIGHT * scale,
        CFG.PHYSICS_ENERGY_WEIGHT * scale,
        CFG.PHYSICS_LAP_WEIGHT * scale
    )

def compute_smooth_energy(embeddings, class_weights, tau=0.1):
    """Smooth energy with LogSumExp (numerically stable)"""
    W_frozen = tf.stop_gradient(class_weights)
    W_norm = tf.nn.l2_normalize(W_frozen, axis=0)
    z_norm = tf.nn.l2_normalize(embeddings, axis=1)
    
    cos_sim = tf.matmul(z_norm, W_norm)
    cos_sim = cos_sim - tf.reduce_max(cos_sim, axis=1, keepdims=True)  # Stability
    
    exp_term = tf.exp(cos_sim / tau)
    log_sum = tf.math.log(tf.reduce_sum(exp_term, axis=1) + 1e-8)
    energy = -tau * log_sum
    
    return energy

def energy_directional_loss(embeddings, class_weights, tau=0.1):
    """Energy gradient directional alignment"""
    with tf.GradientTape(persistent=True) as energy_tape:
        energy_tape.watch(embeddings)
        energy = compute_smooth_energy(embeddings, class_weights, tau)
    
    grad_E = energy_tape.gradient(energy, embeddings)
    del energy_tape
    
    grad_E_norm = tf.nn.l2_normalize(grad_E, axis=1)
    z_norm = tf.nn.l2_normalize(embeddings, axis=1)
    
    cos_sim = tf.reduce_sum(grad_E_norm * z_norm, axis=1)
    cos_sim = tf.clip_by_value(cos_sim, -1.0, 1.0)
    
    loss = tf.reduce_mean(1.0 - cos_sim)
    return loss

# ============================================================================
# TRAINER (with all corrections)
# ============================================================================

class PhysicsConstrainedTrainer(keras.Model):
    """CORRECTED trainer with physics losses only on known samples"""
    
    def __init__(self, base_model, embedding_model, center_loss_layer,
                 unknown_rejection_layer, center_loss_weight, unknown_loss_weight,
                 physics_warmup_epochs, **kwargs):
        super().__init__(**kwargs)
        
        self.base_model = base_model
        self.embedding_model = embedding_model
        self.center_loss_layer = center_loss_layer
        self.unknown_rejection_layer = unknown_rejection_layer
        
        self.center_loss_weight = center_loss_weight
        self.unknown_loss_weight = unknown_loss_weight
        self.physics_warmup_epochs = physics_warmup_epochs
        
        self.current_epoch = tf.Variable(0, trainable=False, dtype=tf.int32)
        
        # Metrics
        self.arcface_loss_tracker = keras.metrics.Mean(name="arcface_loss")
        self.center_loss_tracker = keras.metrics.Mean(name="center_loss")
        self.unknown_loss_tracker = keras.metrics.Mean(name="unknown_loss")
        self.sym_loss_tracker = keras.metrics.Mean(name="sym_loss")
        self.energy_loss_tracker = keras.metrics.Mean(name="energy_loss")
        self.lap_loss_tracker = keras.metrics.Mean(name="lap_loss")
        self.total_loss_tracker = keras.metrics.Mean(name="loss")
        self.accuracy_tracker = keras.metrics.SparseCategoricalAccuracy(name="accuracy")
        self.embed_norm_tracker = keras.metrics.Mean(name="embed_norm")
    
    def call(self, inputs):
        return self.base_model(inputs)
    
    def train_step(self, data):
        inputs, labels = data
        face_input = inputs['face_input']
        
        lambda_sym, lambda_energy, lambda_lap = get_physics_weights_tf(
            self.current_epoch,
            warmup_epochs=self.physics_warmup_epochs
        )
        
        # CRITICAL: Prepare augmented versions OUTSIDE gradient tape
        # (augmentation ops don't have gradients defined)
        face_flipped = tf.image.flip_left_right(face_input)
        # For smoothness loss, use simple operations with gradients
        # Instead of light_augmentation which uses ops without gradients
        
        with tf.GradientTape() as main_tape:
            logits, embeddings = self.base_model(inputs, training=True)
            
            labels = tf.cast(labels, tf.int32)
            known_mask = tf.greater_equal(labels, 0)
            known_embeddings = tf.boolean_mask(embeddings, known_mask)
            
            # ArcFace
            def compute_arcface():
                known_logits = tf.boolean_mask(logits, known_mask)
                known_labels = tf.boolean_mask(labels, known_mask)
                return tf.reduce_mean(
                    tf.nn.sparse_softmax_cross_entropy_with_logits(
                        labels=known_labels,
                        logits=known_logits
                    )
                )
            
            arcface_loss = tf.cond(
                tf.reduce_any(known_mask),
                compute_arcface,
                lambda: tf.constant(0.0, dtype=tf.float32)
            )
            
            # Center loss
            center_loss = self.center_loss_layer(embeddings, labels, training=True)
            
            # Unknown loss
            arcface_layer = self.base_model.get_layer('arcface')
            class_weights = arcface_layer.W
            unknown_loss = self.unknown_rejection_layer(embeddings, labels, class_weights)
            
            # Physics losses (KNOWN SAMPLES ONLY)
            def compute_physics_losses():
                known_face_input = tf.boolean_mask(face_input, known_mask)
                
                # Flip (no gradient needed, use stop_gradient)
                known_flipped = tf.stop_gradient(tf.image.flip_left_right(known_face_input))
                
                # For smoothness, use simple additive noise instead of contrast/brightness
                # This has gradients and won't cause issues
                noise = tf.random.normal(tf.shape(known_face_input), mean=0.0, stddev=0.1)
                known_augmented = known_face_input + noise
                known_augmented = tf.stop_gradient(known_augmented)
                
                embeddings_flip_known = self.embedding_model(known_flipped, training=True)
                embeddings_aug_known = self.embedding_model(known_augmented, training=True)
                
                # L_sym
                diff_sym = known_embeddings - embeddings_flip_known
                sym = tf.reduce_mean(tf.reduce_sum(tf.square(diff_sym), axis=1))
                
                # L_energy
                energy = energy_directional_loss(known_embeddings, class_weights, tau=CFG.ENERGY_TAU)
                
                # L_lap
                diff_lap = known_embeddings - embeddings_aug_known
                lap = tf.reduce_mean(tf.reduce_sum(tf.square(diff_lap), axis=1))
                
                return sym, energy, lap
            
            sym_loss, energy_loss, lap_loss = tf.cond(
                tf.reduce_any(known_mask),
                compute_physics_losses,
                lambda: (tf.constant(0.0, dtype=tf.float32),
                        tf.constant(0.0, dtype=tf.float32),
                        tf.constant(0.0, dtype=tf.float32))
            )
            
            # Total loss
            total_loss = (
                arcface_loss +
                self.center_loss_weight * center_loss +
                self.unknown_loss_weight * unknown_loss +
                lambda_sym * sym_loss +
                lambda_energy * energy_loss +
                lambda_lap * lap_loss
            )
            
            if self.base_model.losses:
                total_loss += tf.add_n(self.base_model.losses)
        
        gradients = main_tape.gradient(total_loss, self.base_model.trainable_variables)
        self.optimizer.apply_gradients(zip(gradients, self.base_model.trainable_variables))
        
        # Update metrics
        self.arcface_loss_tracker.update_state(arcface_loss)
        self.center_loss_tracker.update_state(center_loss)
        self.unknown_loss_tracker.update_state(unknown_loss)
        self.sym_loss_tracker.update_state(sym_loss)
        self.energy_loss_tracker.update_state(energy_loss)
        self.lap_loss_tracker.update_state(lap_loss)
        self.total_loss_tracker.update_state(total_loss)
        
        embed_norms = tf.norm(embeddings, axis=1)
        self.embed_norm_tracker.update_state(tf.reduce_mean(embed_norms))
        
        def update_acc():
            known_logits = tf.boolean_mask(logits, known_mask)
            known_labels = tf.boolean_mask(labels, known_mask)
            self.accuracy_tracker.update_state(known_labels, known_logits)
            return tf.constant(0)
        
        tf.cond(tf.reduce_any(known_mask), update_acc, lambda: tf.constant(0))
        
        return {
            "loss": self.total_loss_tracker.result(),
            "arcface_loss": self.arcface_loss_tracker.result(),
            "center_loss": self.center_loss_tracker.result(),
            "unknown_loss": self.unknown_loss_tracker.result(),
            "sym_loss": self.sym_loss_tracker.result(),
            "energy_loss": self.energy_loss_tracker.result(),
            "lap_loss": self.lap_loss_tracker.result(),
            "accuracy": self.accuracy_tracker.result(),
            "embed_norm": self.embed_norm_tracker.result()
        }
    
    @property
    def metrics(self):
        return [
            self.total_loss_tracker,
            self.arcface_loss_tracker,
            self.center_loss_tracker,
            self.unknown_loss_tracker,
            self.sym_loss_tracker,
            self.energy_loss_tracker,
            self.lap_loss_tracker,
            self.accuracy_tracker,
            self.embed_norm_tracker
        ]

trainer = PhysicsConstrainedTrainer(
    base_model=model,
    embedding_model=embedding_model,
    center_loss_layer=center_loss_layer,
    unknown_rejection_layer=unknown_rejection_layer,
    center_loss_weight=CFG.CENTER_LOSS_WEIGHT,
    unknown_loss_weight=CFG.UNKNOWN_LOSS_WEIGHT,
    physics_warmup_epochs=CFG.PHYSICS_WARMUP_EPOCHS
)

print("\n✓ Trainer initialized (all corrections applied)")

# ============================================================================
# CURRICULUM MIXING
# ============================================================================

def make_curriculum_dataset(known_ds, unknown_ds, unknown_ratio):
    """
    Mix known (face_train) with unknowns (face_val heavily augmented)
    
    CORRECTED: Actually uses unknown_ds (face_val), not slicing from known_ds
    """
    if unknown_ratio == 0.0:
        return known_ds.batch(CFG.BATCH_SIZE).prefetch(CFG.AUTOTUNE)
    
    n_known = max(4, int(CFG.BATCH_SIZE * (1.0 - unknown_ratio)))
    n_unknown = CFG.BATCH_SIZE - n_known
    
    print(f"  Batch: {n_known} known (face_train) + {n_unknown} unknown (face_val augmented)")
    
    # CORRECTED: Separate batching for known and unknown
    known_batched = known_ds.batch(n_known)
    unknown_batched = unknown_ds.batch(n_unknown)
    
    # Zip them together
    mixed_ds = tf.data.Dataset.zip((known_batched, unknown_batched))
    
    def create_mixed_batch(known_batch, unknown_batch):
        known_inputs, known_labels = known_batch
        unknown_inputs, unknown_labels = unknown_batch
        
        # Known faces from face_train
        known_faces = known_inputs['face_input']
        
        # Unknown faces from face_val (heavily augmented)
        unknown_faces_source = unknown_inputs['face_input']
        unknown_faces = tf.map_fn(augment_unknown, unknown_faces_source, dtype=tf.float32)
        unknown_labels_new = tf.fill([tf.shape(unknown_faces)[0]], -1)
        
        # Merge
        merged_faces = tf.concat([known_faces, unknown_faces], axis=0)
        merged_labels = tf.concat([known_labels, unknown_labels_new], axis=0)
        
        # Shuffle
        idx = tf.random.shuffle(tf.range(tf.shape(merged_labels)[0]))
        merged_faces = tf.gather(merged_faces, idx)
        merged_labels = tf.gather(merged_labels, idx)
        
        merged_inputs = {
            'face_input': merged_faces,
            'label_input': merged_labels
        }
        
        return merged_inputs, merged_labels
    
    return mixed_ds.map(create_mixed_batch, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

# ============================================================================
# CALLBACKS
# ============================================================================

class EpochCounterCallback(keras.callbacks.Callback):
    def __init__(self, trainer):
        super().__init__()
        self.trainer = trainer
    
    def on_epoch_begin(self, epoch, logs=None):
        self.trainer.current_epoch.assign(epoch)
        
        lambda_sym, lambda_energy, lambda_lap = get_physics_weights_tf(
            epoch,
            warmup_epochs=self.trainer.physics_warmup_epochs
        )
        print(f"\n  [Epoch {epoch+1}] Physics: "
              f"λ_sym={lambda_sym:.5f}, λ_energy={lambda_energy:.5f}, λ_lap={lambda_lap:.5f}")

# ============================================================================
# TRAINING
# ============================================================================

print("\n" + "=" * 70)
print("TRAINING")
print("=" * 70)

print(f"\nPhase 1 ({CFG.PHASE_1_EPOCHS} epochs): 100% known (face_train)")
print(f"Phase 2 ({CFG.PHASE_2_EPOCHS} epochs): 80% known, 20% unknown (face_val)")
print(f"Phase 3 ({CFG.PHASE_3_EPOCHS} epochs): 60% known, 40% unknown (face_val)")

# CRITICAL: Build model variables before training
# This is required for subclassed models
print("\n✓ Building model variables...")
train_ds_p1_temp = train_data.get_tfdata_unbatched(shuffle=True, augment=True)
dummy_batch = next(iter(train_ds_p1_temp.batch(CFG.BATCH_SIZE)))
_ = trainer(dummy_batch[0], training=False)
print("  Model variables created")

trainer.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3))

# PHASE 1: Only face_train
print("\n" + "=" * 70)
print(f"PHASE 1: Warmup ({CFG.PHASE_1_EPOCHS} epochs)")
print("=" * 70)

train_ds_p1 = train_data.get_tfdata_unbatched(shuffle=True, augment=True)
train_ds_p1 = make_curriculum_dataset(train_ds_p1, None, unknown_ratio=0.0)

phase1_callbacks = [
    EpochCounterCallback(trainer),
    keras.callbacks.ModelCheckpoint(
        'physics_final_phase1.weights.h5',
        monitor='loss',
        save_best_only=True,
        save_weights_only=True,
        verbose=1
    )
]

history_p1 = trainer.fit(
    train_ds_p1,
    steps_per_epoch=CFG.STEPS_PER_EPOCH,
    epochs=CFG.PHASE_1_EPOCHS,
    callbacks=phase1_callbacks
)

# PHASE 2: face_train + face_val (20% unknown)
print("\n" + "=" * 70)
print(f"PHASE 2: Introduction ({CFG.PHASE_2_EPOCHS} epochs)")
print("=" * 70)

train_ds_p2 = train_data.get_tfdata_unbatched(shuffle=True, augment=True)
unknown_ds_p2 = unknown_source_data.get_tfdata_unbatched(shuffle=True, augment=False)
train_ds_p2_mixed = make_curriculum_dataset(train_ds_p2, unknown_ds_p2, unknown_ratio=0.2)

# CRITICAL: Ensure variables exist before loading weights
# (Should already be built from Phase 1, but doing this for safety)
print("\n✓ Ensuring model variables are built...")
dummy_batch_p2 = next(iter(train_ds_p2_mixed))
_ = trainer(dummy_batch_p2[0], training=False)
print("  Variables confirmed")

trainer.load_weights('physics_final_phase1.weights.h5')
print("  Loaded Phase 1 weights")

phase2_callbacks = [
    EpochCounterCallback(trainer),
    keras.callbacks.ModelCheckpoint(
        'physics_final_phase2.weights.h5',
        monitor='loss',
        save_best_only=True,
        save_weights_only=True,
        verbose=1
    )
]

history_p2 = trainer.fit(
    train_ds_p2_mixed,
    steps_per_epoch=CFG.STEPS_PER_EPOCH,
    initial_epoch=CFG.PHASE_1_EPOCHS,
    epochs=CFG.PHASE_1_EPOCHS + CFG.PHASE_2_EPOCHS,
    callbacks=phase2_callbacks
)

# PHASE 3: face_train + face_val (40% unknown, lower LR)
print("\n" + "=" * 70)
print(f"PHASE 3: Hardening ({CFG.PHASE_3_EPOCHS} epochs)")
print("=" * 70)

train_ds_p3 = train_data.get_tfdata_unbatched(shuffle=True, augment=True)
unknown_ds_p3 = unknown_source_data.get_tfdata_unbatched(shuffle=True, augment=False)
train_ds_p3_mixed = make_curriculum_dataset(train_ds_p3, unknown_ds_p3, unknown_ratio=0.4)

# CRITICAL: Ensure variables exist before loading weights
print("\n✓ Ensuring model variables are built...")
dummy_batch_p3 = next(iter(train_ds_p3_mixed))
_ = trainer(dummy_batch_p3[0], training=False)
print("  Variables confirmed")

trainer.load_weights('physics_final_phase2.weights.h5')
print("  Loaded Phase 2 weights")

trainer.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-4))

phase3_callbacks = [
    EpochCounterCallback(trainer),
    keras.callbacks.ModelCheckpoint(
        'physics_final_best.weights.h5',
        monitor='loss',
        save_best_only=True,
        save_weights_only=True,
        verbose=1
    )
]

history_p3 = trainer.fit(
    train_ds_p3_mixed,
    steps_per_epoch=CFG.STEPS_PER_EPOCH,
    initial_epoch=CFG.PHASE_1_EPOCHS + CFG.PHASE_2_EPOCHS,
    epochs=CFG.TOTAL_EPOCHS,
    callbacks=phase3_callbacks
)

print("\n" + "=" * 70)
print("TRAINING COMPLETE")
print("=" * 70)

# ============================================================================
# EXPORT
# ============================================================================

print("\n✓ Exporting inference model...")

trainer.load_weights('physics_final_best.weights.h5')

inference_model = keras.Model(
    inputs=trainer.base_model.input[0],
    outputs=trainer.base_model.output[1],
    name="physics_final_inference"
)

tf.saved_model.save(inference_model, "physics_final_inference_model")
print("  Saved: physics_final_inference_model/")

arcface_layer = model.get_layer('arcface')
class_centers = arcface_layer.W.numpy()
class_centers_norm = class_centers / np.linalg.norm(class_centers, axis=0, keepdims=True)
np.save('physics_final_class_centers.npy', class_centers_norm.T)
print("  Saved: physics_final_class_centers.npy")

metadata = {
    'model_type': 'physics_constrained_final',
    'training_classes': train_data.classes,
    'num_training_classes': train_data.num_classes,
    'unknown_source': 'face_val',
    'test_set': 'face_test (untouched)',
    'corrections': [
        'UnknownRejectionLoss freezes class_weights',
        'CenterLoss updates only during training',
        'Physics losses applied only to known samples',
        'Proper data split: train on face_train, unknowns from face_val'
    ],
    'physics_config': {
        'sym_weight': CFG.PHYSICS_SYM_WEIGHT,
        'energy_weight': CFG.PHYSICS_ENERGY_WEIGHT,
        'lap_weight': CFG.PHYSICS_LAP_WEIGHT,
        'energy_tau': CFG.ENERGY_TAU,
        'warmup_epochs': CFG.PHYSICS_WARMUP_EPOCHS
    }
}

with open('physics_final_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)
print("  Saved: physics_final_metadata.json")

print("\n✓ All done! face_test remains untouched for final evaluation.")
print("=" * 70)

PHYSICS-CONSTRAINED OPEN-SET LIVESTOCK IDENTIFICATION
FINAL CORRECTED VERSION

Data Strategy:
  face_train: Training set (learn identities)
  face_val:   Unknown source (synthetic unknowns during training)
  face_test:  UNTOUCHED (final evaluation only)

LOADING DATA

✓ Training set (face_train):
  345 cows, 3510 images
  Classes: ['DEONI-19-C2082', 'GB-5-C8012', 'GR-6-C2004', 'GRI-4-C25', 'GRI-5-C12']...

✓ Unknown source (face_val):
  30 cows, 291 images
  These cows will serve as synthetic unknowns during training
  Classes: ['HALLIKAR-7-C5027', 'HF-2-C1016', 'HF-2-C2087', 'HF-2-C5042', 'HF-2-C6004']...

✓ Test set (face_test): UNTOUCHED
  Will be used only for final open-set evaluation

BUILDING MODEL

✓ Model built for 345 training classes
  Architecture: EfficientNetB0 → Dense(512) → L2-Normalize → ArcFace
✓ Embedding extraction model created

✓ Trainer initialized (all corrections applied)

TRAINING

Phase 1 (5 epochs): 100% known (face_train)
Phase 2 (15 epochs): 80% known, 20%

INFO:tensorflow:Assets written to: physics_final_inference_model/assets


INFO:tensorflow:Assets written to: physics_final_inference_model/assets


  Saved: physics_final_inference_model/
  Saved: physics_final_class_centers.npy
  Saved: physics_final_metadata.json

✓ All done! face_test remains untouched for final evaluation.


In [8]:
import tensorflow as tf
import numpy as np
import json

"""
DIAGNOSTIC: Why is energy loss stuck at 1.0?

This script loads the trained model and checks:
1. Energy gradient magnitudes
2. Cosine alignment between grad_E and embeddings
3. Energy values distribution
4. Whether the energy loss is actually working
"""

MODEL_PATH = "physics_final_inference_model"
CENTERS_PATH = "physics_final_class_centers.npy"
DATASET_ROOT = "/home/student/Desktop/Kumar/dataset"

print("=" * 80)
print("ENERGY LOSS DIAGNOSTIC")
print("=" * 80)

# Load model
loaded_model = tf.saved_model.load(MODEL_PATH)
infer_fn = loaded_model.signatures["serving_default"]
input_name = list(infer_fn.structured_input_signature[1].keys())[0]
output_name = list(infer_fn.structured_outputs.keys())[0]

# Load centers
class_centers = np.load(CENTERS_PATH)
print(f"✓ Loaded {class_centers.shape[0]} class centers")

# Helper functions
def load_image(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, (224, 224))
    img = tf.cast(img, tf.float32)
    img = tf.keras.applications.efficientnet.preprocess_input(img)
    return img

def compute_smooth_energy(embeddings, class_weights, tau=0.1):
    W_frozen = tf.stop_gradient(class_weights)
    W_norm = tf.nn.l2_normalize(W_frozen, axis=0)
    z_norm = tf.nn.l2_normalize(embeddings, axis=1)
    
    cos_sim = tf.matmul(z_norm, W_norm)
    cos_sim = cos_sim - tf.reduce_max(cos_sim, axis=1, keepdims=True)
    
    exp_term = tf.exp(cos_sim / tau)
    log_sum = tf.math.log(tf.reduce_sum(exp_term, axis=1) + 1e-8)
    energy = -tau * log_sum
    
    return energy

def diagnose_energy_gradient(embeddings, class_weights, tau=0.1):
    """Check energy gradient computation"""
    embeddings_var = tf.Variable(embeddings, dtype=tf.float32)
    
    with tf.GradientTape(persistent=True) as tape:
        tape.watch(embeddings_var)
        energy = compute_smooth_energy(embeddings_var, class_weights, tau)
    
    grad_E = tape.gradient(energy, embeddings_var)
    del tape
    
    # Compute diagnostics
    grad_magnitudes = tf.norm(grad_E, axis=1)
    
    grad_E_norm = tf.nn.l2_normalize(grad_E, axis=1)
    z_norm = tf.nn.l2_normalize(embeddings, axis=1)
    
    cos_similarities = tf.reduce_sum(grad_E_norm * z_norm, axis=1)
    
    directional_loss = 1.0 - tf.reduce_mean(cos_similarities)
    
    return {
        'energy': energy.numpy(),
        'grad_magnitude_mean': tf.reduce_mean(grad_magnitudes).numpy(),
        'grad_magnitude_std': tf.math.reduce_std(grad_magnitudes).numpy(),
        'grad_magnitude_min': tf.reduce_min(grad_magnitudes).numpy(),
        'grad_magnitude_max': tf.reduce_max(grad_magnitudes).numpy(),
        'cosine_mean': tf.reduce_mean(cos_similarities).numpy(),
        'cosine_std': tf.math.reduce_std(cos_similarities).numpy(),
        'directional_loss': directional_loss.numpy()
    }

# Load sample images
import os

def load_sample_images(folder, n=50):
    """Load n sample images from folder"""
    images = []
    for cow_dir in os.listdir(folder)[:10]:  # First 10 cows
        cow_path = os.path.join(folder, cow_dir)
        if not os.path.isdir(cow_path):
            continue
        
        for img_file in os.listdir(cow_path)[:5]:  # 5 images per cow
            if img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
                img_path = os.path.join(cow_path, img_file)
                try:
                    img = load_image(img_path)
                    images.append(img)
                    if len(images) >= n:
                        return images
                except:
                    continue
    return images

print("\n✓ Loading sample images from face_train...")
train_imgs = load_sample_images(os.path.join(DATASET_ROOT, "face_train"), n=50)
train_batch = tf.stack(train_imgs)
print(f"  Loaded {len(train_imgs)} training images")

print("\n✓ Loading sample images from face_test...")
test_imgs = load_sample_images(os.path.join(DATASET_ROOT, "face_test"), n=50)
test_batch = tf.stack(test_imgs)
print(f"  Loaded {len(test_imgs)} test images")

# Extract embeddings
print("\n✓ Extracting embeddings...")
train_output = infer_fn(**{input_name: train_batch})
train_embeddings = train_output[output_name].numpy()

test_output = infer_fn(**{input_name: test_batch})
test_embeddings = test_output[output_name].numpy()

print(f"  Train embeddings shape: {train_embeddings.shape}")
print(f"  Test embeddings shape: {test_embeddings.shape}")

# Diagnose
print("\n" + "=" * 80)
print("DIAGNOSING TRAINING SAMPLES (face_train)")
print("=" * 80)

train_diag = diagnose_energy_gradient(
    tf.constant(train_embeddings, dtype=tf.float32),
    tf.constant(class_centers.T, dtype=tf.float32),
    tau=0.1
)

print(f"\nEnergy:")
print(f"  Mean: {train_diag['energy'].mean():.4f}")
print(f"  Std:  {train_diag['energy'].std():.4f}")
print(f"  Min:  {train_diag['energy'].min():.4f}")
print(f"  Max:  {train_diag['energy'].max():.4f}")

print(f"\nGradient Magnitude:")
print(f"  Mean: {train_diag['grad_magnitude_mean']:.6f}")
print(f"  Std:  {train_diag['grad_magnitude_std']:.6f}")
print(f"  Min:  {train_diag['grad_magnitude_min']:.6f}")
print(f"  Max:  {train_diag['grad_magnitude_max']:.6f}")

if train_diag['grad_magnitude_mean'] < 1e-6:
    print("  ✗ PROBLEM: Gradients are nearly ZERO!")
    print("    Energy loss cannot learn with zero gradients")

print(f"\nCosine Similarity (grad_E, z):")
print(f"  Mean: {train_diag['cosine_mean']:.4f}")
print(f"  Std:  {train_diag['cosine_std']:.4f}")

if abs(train_diag['cosine_mean']) < 0.1:
    print("  ✗ PROBLEM: Gradients are nearly ORTHOGONAL to embeddings!")
    print("    This explains why energy_loss = 1.0 (since 1 - 0 = 1)")

print(f"\nDirectional Loss:")
print(f"  Value: {train_diag['directional_loss']:.4f}")

if train_diag['directional_loss'] > 0.9:
    print("  ✗ PROBLEM: Loss is stuck near 1.0")
    print("    Gradients and embeddings are not aligning")

print("\n" + "=" * 80)
print("DIAGNOSING TEST SAMPLES (face_test - unknowns)")
print("=" * 80)

test_diag = diagnose_energy_gradient(
    tf.constant(test_embeddings, dtype=tf.float32),
    tf.constant(class_centers.T, dtype=tf.float32),
    tau=0.1
)

print(f"\nEnergy:")
print(f"  Mean: {test_diag['energy'].mean():.4f}")
print(f"  Std:  {test_diag['energy'].std():.4f}")
print(f"  Min:  {test_diag['energy'].min():.4f}")
print(f"  Max:  {test_diag['energy'].max():.4f}")

print(f"\nEnergy Comparison:")
print(f"  Train mean: {train_diag['energy'].mean():.4f}")
print(f"  Test mean:  {test_diag['energy'].mean():.4f}")
print(f"  Δ (test - train): {test_diag['energy'].mean() - train_diag['energy'].mean():.4f}")

if test_diag['energy'].mean() < train_diag['energy'].mean():
    print("  ✗ INVERTED: Test has LOWER energy than train")
    print("    Expected: Test should have HIGHER energy")
else:
    print("  ✓ CORRECT: Test has higher energy than train")

print(f"\nGradient Magnitude:")
print(f"  Mean: {test_diag['grad_magnitude_mean']:.6f}")
print(f"  Cosine: {test_diag['cosine_mean']:.4f}")
print(f"  Loss: {test_diag['directional_loss']:.4f}")

# Summary
print("\n" + "=" * 80)
print("DIAGNOSTIC SUMMARY")
print("=" * 80)

issues = []

if train_diag['grad_magnitude_mean'] < 1e-4:
    issues.append("1. Energy gradients are too small (< 1e-4)")

if abs(train_diag['cosine_mean']) < 0.2:
    issues.append("2. Gradients nearly orthogonal to embeddings")

if train_diag['directional_loss'] > 0.9:
    issues.append("3. Directional loss stuck at ~1.0 during training")

if test_diag['energy'].mean() <= train_diag['energy'].mean():
    issues.append("4. Energy separation inverted (test ≤ train)")

if len(issues) > 0:
    print("\n✗ ISSUES FOUND:")
    for issue in issues:
        print(f"  {issue}")
    
    print("\n LIKELY ROOT CAUSE:")
    if train_diag['grad_magnitude_mean'] < 1e-4:
        print("  → Energy gradients vanished during training")
        print("  → This can happen if embeddings are too normalized")
        print("  → Or if tau is too small/large")
    elif abs(train_diag['cosine_mean']) < 0.2:
        print("  → Energy gradient direction is wrong")
        print("  → Gradients should point radially outward from origin")
        print("  → But they're pointing tangentially (orthogonal)")
    
    print("\n RECOMMENDED FIXES:")
    print("  1. Increase energy loss weight from 0.005 to 0.02")
    print("  2. Try different tau (0.05 or 0.2 instead of 0.1)")
    print("  3. Remove directional constraint, use simpler energy loss")
    print("  4. Check if stop_gradient is causing issues")
else:
    print("\n✓ No major issues detected")
    print("  Energy loss should be working correctly")

print("=" * 80)

ENERGY LOSS DIAGNOSTIC
✓ Loaded 345 class centers

✓ Loading sample images from face_train...
  Loaded 50 training images

✓ Loading sample images from face_test...
  Loaded 50 test images

✓ Extracting embeddings...
  Train embeddings shape: (50, 512)
  Test embeddings shape: (50, 512)

DIAGNOSING TRAINING SAMPLES (face_train)

Energy:
  Mean: -0.0181
  Std:  0.0090
  Min:  -0.0429
  Max:  -0.0077

Gradient Magnitude:
  Mean: 0.107362
  Std:  0.058380
  Min:  0.038488
  Max:  0.263310

Cosine Similarity (grad_E, z):
  Mean: 0.0000
  Std:  0.0000
  ✗ PROBLEM: Gradients are nearly ORTHOGONAL to embeddings!
    This explains why energy_loss = 1.0 (since 1 - 0 = 1)

Directional Loss:
  Value: 1.0000
  ✗ PROBLEM: Loss is stuck near 1.0
    Gradients and embeddings are not aligning

DIAGNOSING TEST SAMPLES (face_test - unknowns)

Energy:
  Mean: -0.2807
  Std:  0.0613
  Min:  -0.3905
  Max:  -0.1175

Energy Comparison:
  Train mean: -0.0181
  Test mean:  -0.2807
  Δ (test - train): -0.2626


In [9]:
import tensorflow as tf
import os
import numpy as np
import pandas as pd
from datetime import datetime

"""
Physics-Constrained Model Evaluation on face_test

Data Setup:
- Gallery: face_train (345 cows used in training)
- Known test: face_val (30 cows, used as synthetic unknowns during training)
- Unknown test: face_test (completely unseen, true open-set test)

This evaluation tests:
1. Closed-set accuracy on face_val (different cows, generalization test)
2. Open-set rejection on face_test (true unknowns)
3. Energy-based unknown detection
"""

np.random.seed(42)
tf.random.set_seed(42)

# ============================================================================
# CONFIGURATION
# ============================================================================

DATASET_ROOT = "/home/student/Desktop/Kumar/dataset"
MODEL_PATH = "physics_final_inference_model"
CENTERS_PATH = "physics_final_class_centers.npy"
METADATA_PATH = "physics_final_metadata.json"

class CFG:
    IMG_SIZE = 224
    ENERGY_TAU = 0.1  # Must match training

print("=" * 80)
print("PHYSICS-CONSTRAINED MODEL EVALUATION")
print("=" * 80)
print(f"\nDataset: {DATASET_ROOT}")
print(f"Model:   {MODEL_PATH}")
print(f"Centers: {CENTERS_PATH}")

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def load_image(path):
    """Load and preprocess image"""
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, (CFG.IMG_SIZE, CFG.IMG_SIZE))
    img = tf.cast(img, tf.float32)
    img = tf.keras.applications.efficientnet.preprocess_input(img)
    return img

def compute_energy(embedding, class_centers, tau=0.1):
    """
    Compute energy using LogSumExp (matches training)
    Lower energy = closer to known classes
    Higher energy = far from all known classes (unknown)
    """
    emb_norm = embedding / (np.linalg.norm(embedding) + 1e-8)
    centers_norm = class_centers / (np.linalg.norm(class_centers, axis=1, keepdims=True) + 1e-8)
    
    cos_sim = emb_norm @ centers_norm.T
    cos_sim = cos_sim - np.max(cos_sim)  # Numerical stability
    
    exp_term = np.exp(cos_sim / tau)
    log_sum = np.log(np.sum(exp_term) + 1e-8)
    energy = -tau * log_sum
    
    return energy

# ============================================================================
# LOAD DATA
# ============================================================================

class DatasetLoader:
    """Load cattle dataset from folder"""
    def __init__(self, folder_path):
        self.cattle_data = {}
        
        if not os.path.exists(folder_path):
            raise ValueError(f"Folder not found: {folder_path}")
        
        for cow_id in sorted(os.listdir(folder_path)):
            cow_path = os.path.join(folder_path, cow_id)
            if not os.path.isdir(cow_path):
                continue
            
            imgs = [os.path.join(cow_path, f) for f in os.listdir(cow_path)
                   if f.lower().endswith((".jpg", ".jpeg", ".png"))]
            
            if len(imgs) > 0:
                self.cattle_data[cow_id] = imgs
        
        self.classes = sorted(list(self.cattle_data.keys()))
        self.num_classes = len(self.classes)
        self.total_images = sum(len(v) for v in self.cattle_data.values())

print("\n" + "=" * 80)
print("LOADING DATASETS")
print("=" * 80)

# Load face_train (gallery)
face_train_path = os.path.join(DATASET_ROOT, "face_train")
train_loader = DatasetLoader(face_train_path)
print(f"\n✓ Gallery (face_train): {train_loader.num_classes} cows, {train_loader.total_images} images")

# Load face_val (known test - generalization test)
face_val_path = os.path.join(DATASET_ROOT, "face_val")
val_loader = DatasetLoader(face_val_path)
print(f"✓ Known test (face_val): {val_loader.num_classes} cows, {val_loader.total_images} images")
print(f"  Note: These cows were used as synthetic unknowns during training")

# Load face_test (unknown test - true open-set)
face_test_path = os.path.join(DATASET_ROOT, "face_test")
test_loader = DatasetLoader(face_test_path)
print(f"✓ Unknown test (face_test): {test_loader.num_classes} cows, {test_loader.total_images} images")
print(f"  Note: COMPLETELY UNSEEN during training")

# ============================================================================
# LOAD MODEL & CLASS CENTERS
# ============================================================================

print("\n" + "=" * 80)
print("LOADING MODEL")
print("=" * 80)

# Load model
loaded_model = tf.saved_model.load(MODEL_PATH)
infer_fn = loaded_model.signatures["serving_default"]
input_name = list(infer_fn.structured_input_signature[1].keys())[0]
output_name = list(infer_fn.structured_outputs.keys())[0]
print(f"✓ Model loaded: {MODEL_PATH}")

# Load class centers
class_centers = np.load(CENTERS_PATH)
print(f"✓ Class centers loaded: {class_centers.shape[0]} classes")

def get_embedding(img_path):
    """Extract embedding from image"""
    try:
        img = load_image(img_path)[None]
        output = infer_fn(**{input_name: img})
        emb = output[output_name].numpy()[0]
        # Normalize
        emb = emb / (np.linalg.norm(emb) + 1e-8)
        return emb
    except:
        return None

# ============================================================================
# BUILD GALLERY (from face_train)
# ============================================================================

print("\n" + "=" * 80)
print("BUILDING GALLERY")
print("=" * 80)

print(f"Processing {train_loader.num_classes} training cattle...")

gallery = {}
gallery_stats = []

for cow_id, img_list in train_loader.cattle_data.items():
    embeddings = []
    for img_path in img_list:
        emb = get_embedding(img_path)
        if emb is not None:
            embeddings.append(emb)
    
    if len(embeddings) > 0:
        # Average embedding (prototype)
        avg_emb = np.mean(embeddings, axis=0)
        avg_emb = avg_emb / (np.linalg.norm(avg_emb) + 1e-8)
        gallery[cow_id] = avg_emb
        gallery_stats.append({
            'Cattle_ID': cow_id,
            'Images_Used': len(embeddings)
        })

print(f"✓ Gallery built: {len(gallery)} cattle enrolled")
print(f"  Total images: {sum([s['Images_Used'] for s in gallery_stats])}")

# ============================================================================
# EXTRACT EMBEDDINGS FOR KNOWN TEST (face_val)
# ============================================================================

print("\n" + "=" * 80)
print("PROCESSING KNOWN TEST SET (face_val)")
print("=" * 80)

val_records = []
val_failed = 0

print(f"Processing {val_loader.num_classes} cattle...")

for cow_id, img_list in val_loader.cattle_data.items():
    for img_path in img_list:
        emb = get_embedding(img_path)
        if emb is not None:
            energy = compute_energy(emb, class_centers, CFG.ENERGY_TAU)
            val_records.append((cow_id, img_path, emb, energy))
        else:
            val_failed += 1

print(f"✓ Extracted {len(val_records)} embeddings from {val_loader.num_classes} cattle")
if val_failed > 0:
    print(f"  Failed: {val_failed} images")

# ============================================================================
# EXTRACT EMBEDDINGS FOR UNKNOWN TEST (face_test)
# ============================================================================

print("\n" + "=" * 80)
print("PROCESSING UNKNOWN TEST SET (face_test)")
print("=" * 80)

test_records = []
test_failed = 0

print(f"Processing {test_loader.num_classes} cattle...")

for cow_id, img_list in test_loader.cattle_data.items():
    for img_path in img_list:
        emb = get_embedding(img_path)
        if emb is not None:
            energy = compute_energy(emb, class_centers, CFG.ENERGY_TAU)
            test_records.append((cow_id, img_path, emb, energy))
        else:
            test_failed += 1

print(f"✓ Extracted {len(test_records)} embeddings from {test_loader.num_classes} cattle")
if test_failed > 0:
    print(f"  Failed: {test_failed} images")

# ============================================================================
# ENERGY STATISTICS
# ============================================================================

print("\n" + "=" * 80)
print("ENERGY STATISTICS")
print("=" * 80)

val_energies = np.array([r[3] for r in val_records])
test_energies = np.array([r[3] for r in test_records])

print(f"\n✓ Known test (face_val) energy:")
print(f"  Mean:   {val_energies.mean():.4f}")
print(f"  Std:    {val_energies.std():.4f}")
print(f"  Min:    {val_energies.min():.4f}")
print(f"  Max:    {val_energies.max():.4f}")
print(f"  Median: {np.median(val_energies):.4f}")

print(f"\n✓ Unknown test (face_test) energy:")
print(f"  Mean:   {test_energies.mean():.4f}")
print(f"  Std:    {test_energies.std():.4f}")
print(f"  Min:    {test_energies.min():.4f}")
print(f"  Max:    {test_energies.max():.4f}")
print(f"  Median: {np.median(test_energies):.4f}")

energy_separation = test_energies.mean() - val_energies.mean()
print(f"\n✓ Energy separation (unknown - known): {energy_separation:.4f}")

if energy_separation > 0.3:
    print("  Status: ✓ EXCELLENT separation (>0.3)")
elif energy_separation > 0.1:
    print("  Status: ✓ GOOD separation (0.1-0.3)")
elif energy_separation > 0:
    print("  Status: ⚠ WEAK separation (<0.1)")
else:
    print("  Status: ✗ BAD - unknowns have LOWER energy (inverted!)")

# ============================================================================
# THRESHOLD SWEEP
# ============================================================================

print("\n" + "=" * 80)
print("THRESHOLD SWEEP (Similarity-based)")
print("=" * 80)

thresholds = np.arange(0.50, 0.91, 0.02)  # Broader, finer sweep
threshold_results = []

print("Computing metrics across thresholds...")

for tau in thresholds:
    # Known test metrics
    val_correct = 0
    val_accept = 0
    val_reject = 0
    
    for cow_id, img_path, emb, energy in val_records:
        # Compute similarity to gallery
        scores = {gid: np.dot(emb, gallery[gid]) for gid in gallery}
        best_match = max(scores, key=scores.get)
        best_score = scores[best_match]
        
        if best_score >= tau:
            val_accept += 1
            # Check if correct (but val cows aren't in gallery, so this won't match)
            # We'll compute "would-be accuracy" differently
        else:
            val_reject += 1
    
    # Unknown test metrics
    test_reject = 0
    test_false_accept = 0
    
    for cow_id, img_path, emb, energy in test_records:
        scores = {gid: np.dot(emb, gallery[gid]) for gid in gallery}
        best_match = max(scores, key=scores.get)
        best_score = scores[best_match]
        
        if best_score >= tau:
            test_false_accept += 1
        else:
            test_reject += 1
    
    # Metrics
    far = (test_false_accept / len(test_records)) * 100 if len(test_records) > 0 else 0
    frr = (val_reject / len(val_records)) * 100 if len(val_records) > 0 else 0
    unknown_reject_rate = (test_reject / len(test_records)) * 100 if len(test_records) > 0 else 0
    
    threshold_results.append({
        'threshold': tau,
        'val_accepted': val_accept,
        'val_rejected': val_reject,
        'FRR_%': frr,
        'test_rejected': test_reject,
        'test_false_accepts': test_false_accept,
        'FAR_%': far,
        'unknown_reject_%': unknown_reject_rate
    })

results_df = pd.DataFrame(threshold_results)

print("\n" + "=" * 80)
print("Threshold Sweep Results:")
print("=" * 80)
print("Threshold | FRR (Val) | FAR (Test) | Val Reject | Test Reject")
print("-" * 80)
for _, row in results_df.iterrows():
    print(f"{row['threshold']:.2f}      | {row['FRR_%']:5.1f}%    | "
          f"{row['FAR_%']:6.1f}%    | {row['val_rejected']:4.0f}       | {row['test_rejected']:4.0f}")

# Find best threshold (minimize FAR while keeping FRR reasonable)
# Strategy: lowest FAR with FRR < 50%
reasonable_frr = results_df[results_df['FRR_%'] < 50]
if len(reasonable_frr) > 0:
    optimal_idx = reasonable_frr['FAR_%'].idxmin()
    optimal = results_df.iloc[optimal_idx]
else:
    # Fallback: just minimize FAR
    optimal_idx = results_df['FAR_%'].idxmin()
    optimal = results_df.iloc[optimal_idx]

print("\n" + "=" * 80)
print(f"OPTIMAL THRESHOLD: {optimal['threshold']:.2f}")
print("=" * 80)
print(f"  Known test (face_val):")
print(f"    Accepted:  {optimal['val_accepted']:.0f}/{len(val_records)} ({(optimal['val_accepted']/len(val_records))*100:.1f}%)")
print(f"    Rejected:  {optimal['val_rejected']:.0f}/{len(val_records)} (FRR: {optimal['FRR_%']:.1f}%)")
print(f"\n  Unknown test (face_test):")
print(f"    Rejected:  {optimal['test_rejected']:.0f}/{len(test_records)} ({optimal['unknown_reject_%']:.1f}%)")
print(f"    False accepts: {optimal['test_false_accepts']:.0f}/{len(test_records)} (FAR: {optimal['FAR_%']:.1f}%)")

# ============================================================================
# DETAILED RESULTS AT OPTIMAL THRESHOLD
# ============================================================================

print("\n" + "=" * 80)
print(f"DETAILED ANALYSIS @ Threshold {optimal['threshold']:.2f}")
print("=" * 80)

# Known test detailed
val_detailed = []
for cow_id, img_path, emb, energy in val_records:
    scores = {gid: np.dot(emb, gallery[gid]) for gid in gallery}
    best_match = max(scores, key=scores.get)
    best_score = scores[best_match]
    
    sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    top3 = sorted_scores[:3]
    
    decision = "ACCEPTED" if best_score >= optimal['threshold'] else "REJECTED"
    
    val_detailed.append({
        'Val_Cattle': cow_id,
        'Top1_Match': best_match,
        'Top1_Score': best_score,
        'Top2_Match': top3[1][0] if len(top3) > 1 else "",
        'Top2_Score': top3[1][1] if len(top3) > 1 else 0,
        'Energy': energy,
        'Decision': decision,
        'Image': os.path.basename(img_path)
    })

# Unknown test detailed
test_detailed = []
for cow_id, img_path, emb, energy in test_records:
    scores = {gid: np.dot(emb, gallery[gid]) for gid in gallery}
    best_match = max(scores, key=scores.get)
    best_score = scores[best_match]
    
    sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    top3 = sorted_scores[:3]
    
    decision = "FALSE_ACCEPT" if best_score >= optimal['threshold'] else "CORRECT_REJECT"
    
    test_detailed.append({
        'Test_Cattle': cow_id,
        'Top1_Match': best_match,
        'Top1_Score': best_score,
        'Top2_Match': top3[1][0] if len(top3) > 1 else "",
        'Top2_Score': top3[1][1] if len(top3) > 1 else 0,
        'Energy': energy,
        'Decision': decision,
        'Image': os.path.basename(img_path)
    })

val_df = pd.DataFrame(val_detailed)
test_df = pd.DataFrame(test_detailed)

# Per-cattle summary (test set)
test_per_cattle = test_df.groupby("Test_Cattle").agg(
    Total=("Decision", "count"),
    False_Accepts=("Decision", lambda x: (x == "FALSE_ACCEPT").sum()),
    Correct_Rejects=("Decision", lambda x: (x == "CORRECT_REJECT").sum()),
    FAR_pct=("Decision", lambda x: (x == "FALSE_ACCEPT").sum() / len(x) * 100),
    Avg_Score=("Top1_Score", "mean"),
    Max_Score=("Top1_Score", "max"),
    Avg_Energy=("Energy", "mean")
)

print("\nUnknown test cattle with highest FAR (top 10):")
top_far = test_per_cattle.nlargest(10, 'FAR_pct')
print(top_far[['Total', 'False_Accepts', 'FAR_pct', 'Avg_Score', 'Avg_Energy']].to_string())

print("\nUnknown test cattle with 0% FAR (perfect rejection):")
perfect_reject = test_per_cattle[test_per_cattle['FAR_pct'] == 0.0].head(10)
if len(perfect_reject) > 0:
    print(perfect_reject[['Total', 'Avg_Score', 'Avg_Energy']].to_string())
    print(f"\n  {len(test_per_cattle[test_per_cattle['FAR_pct'] == 0.0])} / {test_loader.num_classes} "
          f"cattle perfectly rejected ({len(test_per_cattle[test_per_cattle['FAR_pct'] == 0.0])/test_loader.num_classes*100:.1f}%)")

# ============================================================================
# SAVE RESULTS
# ============================================================================

print("\n" + "=" * 80)
print("SAVING RESULTS")
print("=" * 80)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
excel_name = f"physics_eval_{timestamp}.xlsx"

with pd.ExcelWriter(excel_name, engine="openpyxl") as writer:
    # Summary
    summary_data = {
        "Metric": [
            "Model Type",
            "Training Cattle (Gallery)",
            "Known Test Cattle (face_val)",
            "Unknown Test Cattle (face_test)",
            "",
            "Optimal Threshold",
            "",
            "Known Test (face_val):",
            "  Accepted",
            "  Rejected (FRR)",
            "",
            "Unknown Test (face_test):",
            "  Correctly Rejected",
            "  False Accepts (FAR)",
            "",
            "Energy Statistics:",
            "  Known mean",
            "  Unknown mean",
            "  Separation (Δ)",
            "",
            "Perfect Rejection:",
            "  Test cattle with 0% FAR"
        ],
        "Value": [
            "Physics-Constrained (Symmetry + Energy + Smoothness)",
            str(train_loader.num_classes),
            str(val_loader.num_classes),
            str(test_loader.num_classes),
            "",
            f"{optimal['threshold']:.2f}",
            "",
            "",
            f"{optimal['val_accepted']:.0f}/{len(val_records)} ({(optimal['val_accepted']/len(val_records))*100:.1f}%)",
            f"{optimal['val_rejected']:.0f}/{len(val_records)} ({optimal['FRR_%']:.1f}%)",
            "",
            "",
            f"{optimal['test_rejected']:.0f}/{len(test_records)} ({optimal['unknown_reject_%']:.1f}%)",
            f"{optimal['test_false_accepts']:.0f}/{len(test_records)} ({optimal['FAR_%']:.1f}%)",
            "",
            "",
            f"{val_energies.mean():.4f}",
            f"{test_energies.mean():.4f}",
            f"{energy_separation:.4f}",
            "",
            "",
            f"{len(test_per_cattle[test_per_cattle['FAR_pct'] == 0.0])}/{test_loader.num_classes} "
            f"({len(test_per_cattle[test_per_cattle['FAR_pct'] == 0.0])/test_loader.num_classes*100:.1f}%)"
        ]
    }
    pd.DataFrame(summary_data).to_excel(writer, sheet_name="Summary", index=False)
    
    # Threshold sweep
    results_df.to_excel(writer, sheet_name="Threshold_Sweep", index=False)
    
    # Val detailed
    val_df.to_excel(writer, sheet_name="Known_Test_Val", index=False)
    
    # Test detailed
    test_df.to_excel(writer, sheet_name="Unknown_Test", index=False)
    
    # Per-cattle
    test_per_cattle.to_excel(writer, sheet_name="Per_Test_Cattle")
    
    # Gallery info
    pd.DataFrame(gallery_stats).to_excel(writer, sheet_name="Gallery_Info", index=False)

print(f"✓ Saved: {excel_name}")

# ============================================================================
# FINAL INTERPRETATION
# ============================================================================

print("\n" + "=" * 80)
print("FINAL INTERPRETATION")
print("=" * 80)

print(f"\n Evaluation Setup:")
print(f"   Gallery:      {train_loader.num_classes} cattle (face_train)")
print(f"   Known test:   {val_loader.num_classes} cattle (face_val, used as unknowns in training)")
print(f"   Unknown test: {test_loader.num_classes} cattle (face_test, completely unseen)")

print(f"\n Performance @ threshold {optimal['threshold']:.2f}:")
print(f"   FAR (false accept rate):     {optimal['FAR_%']:.1f}%")
print(f"   Unknown rejection rate:      {optimal['unknown_reject_%']:.1f}%")
print(f"   Known test rejection (FRR):  {optimal['FRR_%']:.1f}%")

print(f"\n Energy Separation:")
print(f"   Known (val) mean:    {val_energies.mean():.4f}")
print(f"   Unknown (test) mean: {test_energies.mean():.4f}")
print(f"   Δ (separation):      {energy_separation:.4f}")

if optimal['FAR_%'] == 0.0:
    print(f"\n ✓ PERFECT: 0% FAR - All {test_loader.num_classes} unknown cattle correctly rejected!")
elif optimal['FAR_%'] < 5.0:
    print(f"\n ✓ EXCELLENT: FAR = {optimal['FAR_%']:.1f}% (<5%)")
elif optimal['FAR_%'] < 15.0:
    print(f"\n ✓ VERY GOOD: FAR = {optimal['FAR_%']:.1f}% (5-15%)")
elif optimal['FAR_%'] < 30.0:
    print(f"\n ✓ GOOD: FAR = {optimal['FAR_%']:.1f}% (15-30%)")
else:
    print(f"\n ⚠ FAIR: FAR = {optimal['FAR_%']:.1f}% (>30%)")

perfect_count = len(test_per_cattle[test_per_cattle['FAR_pct'] == 0.0])
print(f"\n Perfect rejection: {perfect_count}/{test_loader.num_classes} cattle "
      f"({perfect_count/test_loader.num_classes*100:.1f}%)")

print(f"\n✓ Results saved to: {excel_name}")
print("=" * 80)

PHYSICS-CONSTRAINED MODEL EVALUATION

Dataset: /home/student/Desktop/Kumar/dataset
Model:   physics_final_inference_model
Centers: physics_final_class_centers.npy

LOADING DATASETS

✓ Gallery (face_train): 345 cows, 3510 images
✓ Known test (face_val): 30 cows, 291 images
  Note: These cows were used as synthetic unknowns during training
✓ Unknown test (face_test): 93 cows, 925 images
  Note: COMPLETELY UNSEEN during training

LOADING MODEL
✓ Model loaded: physics_final_inference_model
✓ Class centers loaded: 345 classes

BUILDING GALLERY
Processing 345 training cattle...
✓ Gallery built: 345 cattle enrolled
  Total images: 3510

PROCESSING KNOWN TEST SET (face_val)
Processing 30 cattle...
✓ Extracted 291 embeddings from 30 cattle

PROCESSING UNKNOWN TEST SET (face_test)
Processing 93 cattle...
✓ Extracted 925 embeddings from 93 cattle

ENERGY STATISTICS

✓ Known test (face_val) energy:
  Mean:   -0.3928
  Std:    0.0251
  Min:    -0.4427
  Max:    -0.3021
  Median: -0.3949

✓ Unknown 

In [10]:
import tensorflow as tf
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

"""
THRESHOLD SWEEP ON UNKNOWN TEST SET (face_test)

This script focuses on finding the optimal threshold for rejecting unknowns.
We vary the threshold and compute:
- Number of false accepts
- FAR (False Accept Rate)
- Per-cattle breakdown
- Distribution analysis

Goal: Find threshold that minimizes FAR while being practical.
"""

DATASET_ROOT = "/home/student/Desktop/Kumar/dataset"
MODEL_PATH = "physics_final_inference_model"
CENTERS_PATH = "physics_final_class_centers.npy"

class CFG:
    IMG_SIZE = 224
    ENERGY_TAU = 0.1

print("=" * 80)
print("THRESHOLD SWEEP: UNKNOWN TEST SET (face_test)")
print("=" * 80)

# ============================================================================
# LOAD MODEL & DATA
# ============================================================================

def load_image(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, (CFG.IMG_SIZE, CFG.IMG_SIZE))
    img = tf.cast(img, tf.float32)
    img = tf.keras.applications.efficientnet.preprocess_input(img)
    return img

def compute_energy(embedding, class_centers, tau=0.1):
    emb_norm = embedding / (np.linalg.norm(embedding) + 1e-8)
    centers_norm = class_centers / (np.linalg.norm(class_centers, axis=1, keepdims=True) + 1e-8)
    
    cos_sim = emb_norm @ centers_norm.T
    cos_sim = cos_sim - np.max(cos_sim)
    
    exp_term = np.exp(cos_sim / tau)
    log_sum = np.log(np.sum(exp_term) + 1e-8)
    energy = -tau * log_sum
    
    return energy

class DatasetLoader:
    def __init__(self, folder_path):
        self.cattle_data = {}
        
        for cow_id in sorted(os.listdir(folder_path)):
            cow_path = os.path.join(folder_path, cow_id)
            if not os.path.isdir(cow_path):
                continue
            
            imgs = [os.path.join(cow_path, f) for f in os.listdir(cow_path)
                   if f.lower().endswith((".jpg", ".jpeg", ".png"))]
            
            if len(imgs) > 0:
                self.cattle_data[cow_id] = imgs
        
        self.classes = sorted(list(self.cattle_data.keys()))
        self.num_classes = len(self.classes)
        self.total_images = sum(len(v) for v in self.cattle_data.values())

print("\n✓ Loading datasets...")
face_train_path = os.path.join(DATASET_ROOT, "face_train")
face_test_path = os.path.join(DATASET_ROOT, "face_test")

train_loader = DatasetLoader(face_train_path)
test_loader = DatasetLoader(face_test_path)

print(f"  Gallery (face_train): {train_loader.num_classes} cattle")
print(f"  Unknown test (face_test): {test_loader.num_classes} cattle, {test_loader.total_images} images")

# Load model
loaded_model = tf.saved_model.load(MODEL_PATH)
infer_fn = loaded_model.signatures["serving_default"]
input_name = list(infer_fn.structured_input_signature[1].keys())[0]
output_name = list(infer_fn.structured_outputs.keys())[0]

class_centers = np.load(CENTERS_PATH)
print(f"✓ Model loaded: {class_centers.shape[0]} class centers")

def get_embedding(img_path):
    try:
        img = load_image(img_path)[None]
        output = infer_fn(**{input_name: img})
        emb = output[output_name].numpy()[0]
        emb = emb / (np.linalg.norm(emb) + 1e-8)
        return emb
    except:
        return None

# ============================================================================
# BUILD GALLERY
# ============================================================================

print("\n✓ Building gallery from face_train...")
gallery = {}

for cow_id, img_list in train_loader.cattle_data.items():
    embeddings = []
    for img_path in img_list:
        emb = get_embedding(img_path)
        if emb is not None:
            embeddings.append(emb)
    
    if len(embeddings) > 0:
        avg_emb = np.mean(embeddings, axis=0)
        avg_emb = avg_emb / (np.linalg.norm(avg_emb) + 1e-8)
        gallery[cow_id] = avg_emb

print(f"  {len(gallery)} cattle enrolled")

# ============================================================================
# EXTRACT UNKNOWN TEST EMBEDDINGS
# ============================================================================

print("\n✓ Processing unknown test set (face_test)...")

test_records = []

for cow_id, img_list in test_loader.cattle_data.items():
    for img_path in img_list:
        emb = get_embedding(img_path)
        if emb is not None:
            # Compute similarity to gallery
            scores = {gid: np.dot(emb, gallery[gid]) for gid in gallery}
            best_match = max(scores, key=scores.get)
            best_score = scores[best_match]
            
            # Compute energy
            energy = compute_energy(emb, class_centers, CFG.ENERGY_TAU)
            
            test_records.append({
                'cattle_id': cow_id,
                'image_path': img_path,
                'best_match': best_match,
                'similarity': best_score,
                'energy': energy
            })

print(f"  {len(test_records)} test images processed")

# Convert to DataFrame
test_df = pd.DataFrame(test_records)

# ============================================================================
# THRESHOLD SWEEP (SIMILARITY-BASED)
# ============================================================================

print("\n" + "=" * 80)
print("THRESHOLD SWEEP: SIMILARITY-BASED REJECTION")
print("=" * 80)

thresholds_sim = np.arange(0.50, 0.91, 0.01)  # Fine-grained sweep
similarity_results = []

for tau in thresholds_sim:
    rejected = (test_df['similarity'] < tau).sum()
    false_accepts = (test_df['similarity'] >= tau).sum()
    far = (false_accepts / len(test_df)) * 100
    reject_rate = (rejected / len(test_df)) * 100
    
    similarity_results.append({
        'threshold': tau,
        'false_accepts': false_accepts,
        'rejected': rejected,
        'FAR_%': far,
        'reject_rate_%': reject_rate,
        'avg_similarity': test_df['similarity'].mean(),
        'median_similarity': test_df['similarity'].median()
    })

sim_results_df = pd.DataFrame(similarity_results)

print("\nSimilarity-based Thresholding:")
print("=" * 80)
print("Threshold | False Accepts | Rejected | FAR    | Reject Rate")
print("-" * 80)

for _, row in sim_results_df.iloc[::2].iterrows():  # Every 2nd row for readability
    print(f"{row['threshold']:.2f}      | {row['false_accepts']:4.0f}          | "
          f"{row['rejected']:4.0f}     | {row['FAR_%']:5.1f}% | {row['reject_rate_%']:5.1f}%")

# Find key operating points
best_far_idx = sim_results_df['FAR_%'].idxmin()
best_far = sim_results_df.iloc[best_far_idx]

# Find threshold for specific FAR targets
far_5_idx = sim_results_df[sim_results_df['FAR_%'] <= 5.0].index
far_10_idx = sim_results_df[sim_results_df['FAR_%'] <= 10.0].index
far_20_idx = sim_results_df[sim_results_df['FAR_%'] <= 20.0].index

print("\n" + "=" * 80)
print("KEY OPERATING POINTS")
print("=" * 80)

print(f"\n1. MINIMUM FAR:")
print(f"   Threshold: {best_far['threshold']:.2f}")
print(f"   FAR: {best_far['FAR_%']:.2f}%")
print(f"   Rejected: {best_far['rejected']:.0f}/{len(test_df)} ({best_far['reject_rate_%']:.1f}%)")

if len(far_5_idx) > 0:
    far_5 = sim_results_df.iloc[far_5_idx[0]]
    print(f"\n2. FAR ≤ 5%:")
    print(f"   Threshold: {far_5['threshold']:.2f}")
    print(f"   FAR: {far_5['FAR_%']:.2f}%")
    print(f"   Rejected: {far_5['rejected']:.0f}/{len(test_df)} ({far_5['reject_rate_%']:.1f}%)")
else:
    print(f"\n2. FAR ≤ 5%: Not achievable")

if len(far_10_idx) > 0:
    far_10 = sim_results_df.iloc[far_10_idx[0]]
    print(f"\n3. FAR ≤ 10%:")
    print(f"   Threshold: {far_10['threshold']:.2f}")
    print(f"   FAR: {far_10['FAR_%']:.2f}%")
    print(f"   Rejected: {far_10['rejected']:.0f}/{len(test_df)} ({far_10['reject_rate_%']:.1f}%)")
else:
    print(f"\n3. FAR ≤ 10%: Not achievable")

if len(far_20_idx) > 0:
    far_20 = sim_results_df.iloc[far_20_idx[0]]
    print(f"\n4. FAR ≤ 20%:")
    print(f"   Threshold: {far_20['threshold']:.2f}")
    print(f"   FAR: {far_20['FAR_%']:.2f}%")
    print(f"   Rejected: {far_20['rejected']:.0f}/{len(test_df)} ({far_20['reject_rate_%']:.1f}%)")
else:
    print(f"\n4. FAR ≤ 20%: Not achievable")

# ============================================================================
# THRESHOLD SWEEP (ENERGY-BASED)
# ============================================================================

print("\n" + "=" * 80)
print("THRESHOLD SWEEP: ENERGY-BASED REJECTION")
print("=" * 80)

# Energy thresholds (higher energy = more likely unknown)
energy_percentiles = np.percentile(test_df['energy'], np.arange(5, 100, 5))
thresholds_energy = np.linspace(test_df['energy'].min(), test_df['energy'].max(), 30)

energy_results = []

for tau in thresholds_energy:
    rejected = (test_df['energy'] > tau).sum()
    false_accepts = (test_df['energy'] <= tau).sum()
    far = (false_accepts / len(test_df)) * 100
    reject_rate = (rejected / len(test_df)) * 100
    
    energy_results.append({
        'threshold': tau,
        'false_accepts': false_accepts,
        'rejected': rejected,
        'FAR_%': far,
        'reject_rate_%': reject_rate
    })

energy_results_df = pd.DataFrame(energy_results)

print("\nEnergy-based Thresholding:")
print("=" * 80)
print("Energy Threshold | False Accepts | Rejected | FAR    | Reject Rate")
print("-" * 80)

for _, row in energy_results_df.iloc[::3].iterrows():  # Every 3rd row
    print(f"{row['threshold']:7.4f}      | {row['false_accepts']:4.0f}          | "
          f"{row['rejected']:4.0f}     | {row['FAR_%']:5.1f}% | {row['reject_rate_%']:5.1f}%")

# Find best energy threshold
best_energy_idx = energy_results_df['FAR_%'].idxmin()
best_energy = energy_results_df.iloc[best_energy_idx]

print(f"\nBest Energy Threshold:")
print(f"   Threshold: {best_energy['threshold']:.4f}")
print(f"   FAR: {best_energy['FAR_%']:.2f}%")
print(f"   Rejected: {best_energy['rejected']:.0f}/{len(test_df)} ({best_energy['reject_rate_%']:.1f}%)")

# ============================================================================
# PER-CATTLE ANALYSIS
# ============================================================================

print("\n" + "=" * 80)
print("PER-CATTLE BREAKDOWN")
print("=" * 80)

per_cattle_analysis = []

for cattle_id in test_df['cattle_id'].unique():
    cattle_data = test_df[test_df['cattle_id'] == cattle_id]
    
    # At best similarity threshold
    false_accepts_sim = (cattle_data['similarity'] >= best_far['threshold']).sum()
    
    # At best energy threshold
    false_accepts_energy = (cattle_data['energy'] <= best_energy['threshold']).sum()
    
    per_cattle_analysis.append({
        'Cattle_ID': cattle_id,
        'Total_Images': len(cattle_data),
        'Avg_Similarity': cattle_data['similarity'].mean(),
        'Max_Similarity': cattle_data['similarity'].max(),
        'Avg_Energy': cattle_data['energy'].mean(),
        'FA_Similarity': false_accepts_sim,
        'FA_Energy': false_accepts_energy,
        'FAR_Similarity_%': (false_accepts_sim / len(cattle_data)) * 100,
        'FAR_Energy_%': (false_accepts_energy / len(cattle_data)) * 100
    })

per_cattle_df = pd.DataFrame(per_cattle_analysis)

print("\nCattle with highest FAR (similarity-based, top 10):")
top_far_sim = per_cattle_df.nlargest(10, 'FAR_Similarity_%')
print(top_far_sim[['Cattle_ID', 'Total_Images', 'FA_Similarity', 'FAR_Similarity_%', 'Avg_Similarity']].to_string(index=False))

print("\nCattle with 0% FAR (perfect rejection):")
perfect_reject = per_cattle_df[per_cattle_df['FAR_Similarity_%'] == 0]
print(f"  {len(perfect_reject)} / {test_loader.num_classes} cattle ({len(perfect_reject)/test_loader.num_classes*100:.1f}%)")
if len(perfect_reject) > 0:
    print(perfect_reject[['Cattle_ID', 'Total_Images', 'Avg_Similarity', 'Avg_Energy']].head(10).to_string(index=False))

# ============================================================================
# VISUALIZATION
# ============================================================================

print("\n" + "=" * 80)
print("GENERATING VISUALIZATIONS")
print("=" * 80)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Similarity-based FAR vs Threshold
ax1 = axes[0, 0]
ax1.plot(sim_results_df['threshold'], sim_results_df['FAR_%'], 'b-', linewidth=2)
ax1.axhline(y=5, color='green', linestyle='--', alpha=0.5, label='5% FAR')
ax1.axhline(y=10, color='orange', linestyle='--', alpha=0.5, label='10% FAR')
ax1.axhline(y=20, color='red', linestyle='--', alpha=0.5, label='20% FAR')
ax1.axvline(x=best_far['threshold'], color='black', linestyle=':', alpha=0.7, 
           label=f'Best ({best_far["threshold"]:.2f})')
ax1.set_xlabel('Similarity Threshold')
ax1.set_ylabel('False Accept Rate (%)')
ax1.set_title('FAR vs Similarity Threshold')
ax1.grid(True, alpha=0.3)
ax1.legend()

# Plot 2: Energy-based FAR vs Threshold
ax2 = axes[0, 1]
ax2.plot(energy_results_df['threshold'], energy_results_df['FAR_%'], 'r-', linewidth=2)
ax2.axhline(y=5, color='green', linestyle='--', alpha=0.5, label='5% FAR')
ax2.axhline(y=10, color='orange', linestyle='--', alpha=0.5, label='10% FAR')
ax2.axvline(x=best_energy['threshold'], color='black', linestyle=':', alpha=0.7,
           label=f'Best ({best_energy["threshold"]:.3f})')
ax2.set_xlabel('Energy Threshold')
ax2.set_ylabel('False Accept Rate (%)')
ax2.set_title('FAR vs Energy Threshold')
ax2.grid(True, alpha=0.3)
ax2.legend()

# Plot 3: Similarity distribution
ax3 = axes[1, 0]
ax3.hist(test_df['similarity'], bins=50, color='blue', alpha=0.7, edgecolor='black')
ax3.axvline(x=best_far['threshold'], color='red', linestyle='--', linewidth=2,
           label=f'Best threshold ({best_far["threshold"]:.2f})')
ax3.set_xlabel('Similarity Score')
ax3.set_ylabel('Frequency')
ax3.set_title('Distribution of Similarity Scores (Unknown Test Set)')
ax3.legend()
ax3.grid(True, alpha=0.3)

# Plot 4: Energy distribution
ax4 = axes[1, 1]
ax4.hist(test_df['energy'], bins=50, color='red', alpha=0.7, edgecolor='black')
ax4.axvline(x=best_energy['threshold'], color='blue', linestyle='--', linewidth=2,
           label=f'Best threshold ({best_energy["threshold"]:.3f})')
ax4.set_xlabel('Energy')
ax4.set_ylabel('Frequency')
ax4.set_title('Distribution of Energy (Unknown Test Set)')
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('unknown_threshold_sweep.png', dpi=300, bbox_inches='tight')
print("✓ Saved: unknown_threshold_sweep.png")
plt.close()

# ============================================================================
# SAVE RESULTS
# ============================================================================

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
excel_name = f"unknown_threshold_sweep_{timestamp}.xlsx"

with pd.ExcelWriter(excel_name, engine='openpyxl') as writer:
    # Similarity sweep
    sim_results_df.to_excel(writer, sheet_name='Similarity_Sweep', index=False)
    
    # Energy sweep
    energy_results_df.to_excel(writer, sheet_name='Energy_Sweep', index=False)
    
    # Per-cattle
    per_cattle_df.to_excel(writer, sheet_name='Per_Cattle', index=False)
    
    # Summary
    summary_data = {
        'Metric': [
            'Total Unknown Cattle',
            'Total Unknown Images',
            '',
            'SIMILARITY-BASED:',
            'Best Threshold',
            'Minimum FAR',
            'Rejection Rate',
            '',
            'ENERGY-BASED:',
            'Best Threshold',
            'Minimum FAR',
            'Rejection Rate',
            '',
            'Cattle with 0% FAR',
            'Cattle with 100% FAR'
        ],
        'Value': [
            str(test_loader.num_classes),
            str(len(test_df)),
            '',
            '',
            f"{best_far['threshold']:.2f}",
            f"{best_far['FAR_%']:.2f}%",
            f"{best_far['reject_rate_%']:.2f}%",
            '',
            '',
            f"{best_energy['threshold']:.4f}",
            f"{best_energy['FAR_%']:.2f}%",
            f"{best_energy['reject_rate_%']:.2f}%",
            '',
            f"{len(perfect_reject)} ({len(perfect_reject)/test_loader.num_classes*100:.1f}%)",
            f"{len(per_cattle_df[per_cattle_df['FAR_Similarity_%'] == 100])} ({len(per_cattle_df[per_cattle_df['FAR_Similarity_%'] == 100])/test_loader.num_classes*100:.1f}%)"
        ]
    }
    pd.DataFrame(summary_data).to_excel(writer, sheet_name='Summary', index=False)

print(f"✓ Saved: {excel_name}")

# ============================================================================
# FINAL SUMMARY
# ============================================================================

print("\n" + "=" * 80)
print("THRESHOLD SWEEP SUMMARY")
print("=" * 80)

print(f"\nDataset: {test_loader.num_classes} unknown cattle, {len(test_df)} images")

print(f"\nSIMILARITY-BASED REJECTION:")
print(f"  Best threshold: {best_far['threshold']:.2f}")
print(f"  Minimum FAR: {best_far['FAR_%']:.2f}%")
print(f"  Rejection rate: {best_far['reject_rate_%']:.2f}%")

print(f"\nENERGY-BASED REJECTION:")
print(f"  Best threshold: {best_energy['threshold']:.4f}")
print(f"  Minimum FAR: {best_energy['FAR_%']:.2f}%")
print(f"  Rejection rate: {best_energy['reject_rate_%']:.2f}%")

print(f"\nPERFECT REJECTION:")
print(f"  {len(perfect_reject)} / {test_loader.num_classes} cattle ({len(perfect_reject)/test_loader.num_classes*100:.1f}%)")

print("\n✓ Analysis complete!")
print("=" * 80)

THRESHOLD SWEEP: UNKNOWN TEST SET (face_test)

✓ Loading datasets...
  Gallery (face_train): 345 cattle
  Unknown test (face_test): 93 cattle, 925 images
✓ Model loaded: 345 class centers

✓ Building gallery from face_train...
  345 cattle enrolled

✓ Processing unknown test set (face_test)...
  925 test images processed

THRESHOLD SWEEP: SIMILARITY-BASED REJECTION

Similarity-based Thresholding:
Threshold | False Accepts | Rejected | FAR    | Reject Rate
--------------------------------------------------------------------------------
0.50      |  921          |    4     |  99.6% |   0.4%
0.52      |  913          |   12     |  98.7% |   1.3%
0.54      |  896          |   29     |  96.9% |   3.1%
0.56      |  846          |   79     |  91.5% |   8.5%
0.58      |  750          |  175     |  81.1% |  18.9%
0.60      |  598          |  327     |  64.6% |  35.4%
0.62      |  419          |  506     |  45.3% |  54.7%
0.64      |  251          |  674     |  27.1% |  72.9%
0.66      |  130   

In [2]:
import tensorflow as tf
import os
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers
import json

"""
PHYSICS-CONSTRAINED OPEN-SET LIVESTOCK IDENTIFICATION
FINAL VERSION (Energy Loss Removed)

Final Objective Function:
L = L_arc + λ1·L_center + λ2·L_unknown + λ3·L_sym + λ4·L_lap

Components:
- L_arc: ArcFace angular margin loss
- L_center: Center loss for intra-class compactness
- L_unknown: Unknown rejection loss
- L_sym: Symmetry constraint (flip invariance)
- L_lap: Smoothness constraint (perturbation robustness)

Note on Energy Loss:
Originally attempted L_energy (directional energy gradient alignment).
Removed due to geometric incompatibility: L2-normalized embeddings lie on
a unit hypersphere where energy gradients are necessarily tangent, making
directional alignment impossible (cos(∇E, z) = 0 by construction).

Data Strategy:
- face_train (345 cows): Training set - learn discriminative embeddings
- face_val (30 cows): Synthetic unknown source during curriculum training
- face_test: COMPLETELY UNTOUCHED - used only for final open-set evaluation

Training Protocol:
- Phase 1: Train on face_train only (100% known, physics warmup)
- Phase 2-3: Mix face_train (known) + face_val (as unknowns via heavy augmentation)
- Validation: Monitor on face_train subset (not face_val)
- Evaluation: face_test as true unknowns

Key Corrections:
1. UnknownRejectionLoss freezes class_weights (no gradient leak)
2. CenterLoss updates only during training (no data leakage)
3. Physics losses apply only to known samples (cleaner interpretation)
4. Proper data split: train on face_train, val serves as unknown source
5. Energy loss removed (geometrically incompatible)
"""

# Set seeds
np.random.seed(42)
tf.random.set_seed(42)

# GPU setup
gpus = tf.config.experimental.list_physical_devices('GPU')
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except:
        pass

dataset_root = "/home/student/Desktop/Kumar/dataset"

# ============================================================================
# CONFIGURATION
# ============================================================================

class CFG:
    IMG_SIZE = 224
    EMBED_DIM = 512
    MARGIN = 0.5
    SCALE = 64
    BATCH_SIZE = 32
    
    # Loss weights
    CENTER_LOSS_WEIGHT = 0.0001
    UNKNOWN_LOSS_WEIGHT = 0.1
    
    # Physics loss weights (final values after warmup)
    PHYSICS_SYM_WEIGHT = 0.01      # L_sym - Symmetry
    PHYSICS_ENERGY_WEIGHT = 0.0    # L_energy - REMOVED (geometrically incompatible)
    PHYSICS_LAP_WEIGHT = 0.01      # L_lap - Smoothness
    
    # Physics warmup
    PHYSICS_WARMUP_EPOCHS = 10
    
    # Energy temperature
    ENERGY_TAU = 0.1
    
    STEPS_PER_EPOCH = 150
    AUTOTUNE = tf.data.AUTOTUNE
    
    # Curriculum schedule
    PHASE_1_EPOCHS = 5   # Warmup: 100% known from face_train
    PHASE_2_EPOCHS = 15  # Introduction: 80% known, 20% unknown (from face_val)
    PHASE_3_EPOCHS = 10  # Hardening: 60% known, 40% unknown (from face_val)
    
    TOTAL_EPOCHS = PHASE_1_EPOCHS + PHASE_2_EPOCHS + PHASE_3_EPOCHS

print("=" * 70)
print("PHYSICS-CONSTRAINED OPEN-SET LIVESTOCK IDENTIFICATION")
print("FINAL VERSION (Energy Loss Removed)")
print("=" * 70)
print(f"\nFinal Loss Function:")
print(f"  L = L_arc + λ1·L_center + λ2·L_unknown + λ3·L_sym + λ4·L_lap")
print(f"\nData Strategy:")
print(f"  face_train: Training set (learn identities)")
print(f"  face_val:   Unknown source (synthetic unknowns during training)")
print(f"  face_test:  UNTOUCHED (final evaluation only)")
print(f"\nNote: Energy loss removed due to geometric incompatibility")

# ============================================================================
# DATA LOADING
# ============================================================================

def load_image(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, (CFG.IMG_SIZE, CFG.IMG_SIZE))
    img = tf.cast(img, tf.float32)
    img = tf.keras.applications.efficientnet.preprocess_input(img)
    return img

def augment_image(img):
    """Standard augmentation for known samples"""
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, 0.25)
    img = tf.image.random_contrast(img, 0.8, 1.2)
    img = tf.image.random_saturation(img, 0.8, 1.2)
    return img

def light_augmentation(images):
    """Light augmentation for L_lap (batch-safe)"""
    images = tf.image.random_brightness(images, 0.15)
    images = tf.image.random_contrast(images, 0.9, 1.1)
    return images

def augment_unknown(img):
    """Heavy augmentation to create synthetic unknowns from face_val"""
    img = tf.image.random_brightness(img, 0.5)
    img = tf.image.random_contrast(img, 0.5, 1.5)
    img = tf.image.random_saturation(img, 0.5, 1.5)
    img = tf.image.random_hue(img, 0.2)
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    img = tf.image.rot90(img, k=tf.random.uniform([], 0, 4, dtype=tf.int32))
    noise = tf.random.normal(shape=tf.shape(img), mean=0.0, stddev=0.1)
    img = img + noise
    img = tf.clip_by_value(img, -2.0, 2.0)
    return img

class DatasetBuilder:
    """Load cow images from a single folder"""
    def __init__(self, folder_path, seed=42):
        self.cow_data = {}
        np.random.seed(seed)
        
        if not os.path.exists(folder_path):
            raise ValueError(f"Folder not found: {folder_path}")
        
        for cow_id in sorted(os.listdir(folder_path)):
            cow_path = os.path.join(folder_path, cow_id)
            if not os.path.isdir(cow_path):
                continue
            
            imgs = [os.path.join(cow_path, f) for f in os.listdir(cow_path)
                   if f.lower().endswith((".jpg", ".jpeg", ".png"))]
            
            if len(imgs) > 0:
                self.cow_data[cow_id] = imgs
        
        self.classes = sorted(list(self.cow_data.keys()))
        self.cls_to_idx = {c: i for i, c in enumerate(self.classes)}
        self.num_classes = len(self.classes)
        self.total_images = sum(len(v) for v in self.cow_data.values())
    
    def get_tfdata_unbatched(self, shuffle=True, augment=False):
        """Get unbatched dataset for this folder"""
        face_paths = []
        labels = []
        
        for cow_id, face_list in self.cow_data.items():
            for face_path in face_list:
                face_paths.append(face_path)
                labels.append(self.cls_to_idx[cow_id])
        
        ds = tf.data.Dataset.from_tensor_slices({
            'face_path': face_paths,
            'label': labels
        })
        
        def map_fn(sample):
            face_img = load_image(sample['face_path'])
            
            if augment:
                face_img = augment_image(face_img)
            
            return {
                'face_input': face_img,
                'label_input': sample['label']
            }, sample['label']
        
        ds = ds.map(map_fn, num_parallel_calls=tf.data.AUTOTUNE)
        
        if shuffle:
            ds = ds.shuffle(2000, seed=42, reshuffle_each_iteration=True)
        
        ds = ds.repeat()
        return ds

# Load data
print("\n" + "=" * 70)
print("LOADING DATA")
print("=" * 70)

face_train_path = os.path.join(dataset_root, "face_train")
face_val_path = os.path.join(dataset_root, "face_val")

train_data = DatasetBuilder(face_train_path)
unknown_source_data = DatasetBuilder(face_val_path)  # Will be used as unknowns

print(f"\n✓ Training set (face_train):")
print(f"  {train_data.num_classes} cows, {train_data.total_images} images")
print(f"  Classes: {train_data.classes[:5]}...")

print(f"\n✓ Unknown source (face_val):")
print(f"  {unknown_source_data.num_classes} cows, {unknown_source_data.total_images} images")
print(f"  These cows will serve as synthetic unknowns during training")
print(f"  Classes: {unknown_source_data.classes[:5]}...")

print(f"\n✓ Test set (face_test): UNTOUCHED")
print(f"  Will be used only for final open-set evaluation")

# ============================================================================
# MODEL ARCHITECTURE
# ============================================================================

def build_model(num_classes, embedding_size=512, margin=0.5, scale=64):
    """Build model with named embedding layer"""
    face_input = keras.Input(shape=(224, 224, 3), name="face_input")
    label_input = keras.Input(shape=(), dtype=tf.int32, name="label_input")
    
    backbone = keras.applications.EfficientNetB0(
        include_top=False,
        input_shape=(224, 224, 3),
        weights="imagenet"
    )
    
    for layer in backbone.layers[:-30]:
        layer.trainable = False
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False
    
    x = backbone(face_input, training=True)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.4)(x)
    
    x = layers.Dense(embedding_size, kernel_regularizer=keras.regularizers.l2(1e-4),
                     name="embedding_dense")(x)
    
    embeddings = layers.Lambda(
        lambda x: tf.math.l2_normalize(x, axis=1),
        name="l2_normalize"
    )(x)
    
    class SafeArcFaceLayer(layers.Layer):
        def __init__(self, n_classes, margin=0.5, scale=64, **kwargs):
            super().__init__(**kwargs)
            self.n_classes = n_classes
            self.margin = margin
            self.scale = scale
            self.cos_m = tf.cos(margin)
            self.sin_m = tf.sin(margin)
        
        def build(self, input_shape):
            self.W = self.add_weight(
                name="arcface_weights",
                shape=(input_shape[0][-1], self.n_classes),
                initializer=keras.initializers.HeNormal(),
                trainable=True,
                regularizer=keras.regularizers.l2(1e-4)
            )
            super().build(input_shape)
        
        def call(self, inputs):
            embeddings, labels = inputs
            
            W_norm = tf.nn.l2_normalize(self.W, axis=0)
            embeddings_norm = tf.nn.l2_normalize(embeddings, axis=1)
            
            cos_theta = tf.matmul(embeddings_norm, W_norm)
            cos_theta = tf.clip_by_value(cos_theta, -1.0 + 1e-7, 1.0 - 1e-7)
            
            sin_theta = tf.sqrt(tf.maximum(1.0 - tf.square(cos_theta), 1e-7))
            target_logits = cos_theta * self.cos_m - sin_theta * self.sin_m
            
            labels = tf.cast(labels, tf.int32)
            known_mask = tf.greater_equal(labels, 0)
            labels_safe = tf.where(known_mask, labels, tf.zeros_like(labels))
            
            onehot_raw = tf.one_hot(labels_safe, depth=self.n_classes, dtype=tf.float32)
            onehot = onehot_raw * tf.expand_dims(tf.cast(known_mask, tf.float32), axis=1)
            
            output = cos_theta * (1.0 - onehot) + target_logits * onehot
            output = output * self.scale
            
            return output
        
        def get_config(self):
            return {"n_classes": self.n_classes, "margin": self.margin, "scale": self.scale}
    
    logits = SafeArcFaceLayer(num_classes, margin=margin, scale=scale,
                              name="arcface")([embeddings, label_input])
    
    model = keras.Model(
        inputs=[face_input, label_input],
        outputs=[logits, embeddings],
        name="physics_constrained_model"
    )
    
    return model

print("\n" + "=" * 70)
print("BUILDING MODEL")
print("=" * 70)

# CRITICAL: Model has num_classes = face_train only (345 cows)
model = build_model(
    train_data.num_classes,  # Only train_data classes!
    embedding_size=CFG.EMBED_DIM,
    margin=CFG.MARGIN,
    scale=CFG.SCALE
)

print(f"\n✓ Model built for {train_data.num_classes} training classes")
print(f"  Architecture: EfficientNetB0 → Dense(512) → L2-Normalize → ArcFace")

# Embedding-only model
embedding_model = keras.Model(
    inputs=model.input[0],
    outputs=model.get_layer("l2_normalize").output,
    name="embedding_extractor"
)

print(f"✓ Embedding extraction model created")

# ============================================================================
# LOSSES (with all corrections)
# ============================================================================

class CenterLossLayer(layers.Layer):
    """CORRECTED: Only update centers during training"""
    def __init__(self, num_classes, embedding_dim, alpha=0.5, **kwargs):
        super().__init__(**kwargs)
        self.num_classes = num_classes
        self.embedding_dim = embedding_dim
        self.alpha = tf.constant(alpha, dtype=tf.float32)
    
    def build(self, input_shape):
        self.centers = self.add_weight(
            name="class_centers",
            shape=(self.num_classes, self.embedding_dim),
            initializer="zeros",
            trainable=False
        )
        super().build(input_shape)
    
    def call(self, embeddings, labels, training=True):
        labels = tf.cast(labels, tf.int32)
        known_mask = labels >= 0
        known_embeddings = tf.boolean_mask(embeddings, known_mask)
        known_labels = tf.boolean_mask(labels, known_mask)
        
        if tf.shape(known_embeddings)[0] == 0:
            return tf.constant(0.0, dtype=tf.float32)
        
        centers_batch = tf.gather(self.centers, known_labels)
        diff = known_embeddings - centers_batch
        loss = tf.reduce_mean(tf.reduce_sum(tf.square(diff), axis=1))
        
        if not training:
            return loss
        
        # Update centers (only during training)
        unique_labels, idx = tf.unique(known_labels)
        num_unique = tf.shape(unique_labels)[0]
        summed = tf.math.unsorted_segment_sum(known_embeddings, idx, num_unique)
        counts = tf.math.unsorted_segment_sum(tf.ones_like(idx, dtype=tf.float32), idx, num_unique)
        counts = tf.reshape(counts, (-1, 1))
        new_centers = summed / (counts + 1e-8)
        
        current_centers = tf.gather(self.centers, unique_labels)
        updated = self.alpha * current_centers + (1.0 - self.alpha) * new_centers
        
        scatter_idx = tf.expand_dims(unique_labels, axis=1)
        self.centers.assign(tf.tensor_scatter_nd_update(self.centers, scatter_idx, updated))
        
        return loss

class UnknownRejectionLoss(layers.Layer):
    """CORRECTED: Freeze class_weights"""
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
    
    def call(self, embeddings, labels, class_weights):
        labels = tf.cast(labels, tf.int32)
        unknown_mask = tf.less(labels, 0)
        unknown_embeddings = tf.boolean_mask(embeddings, unknown_mask)
        
        if tf.shape(unknown_embeddings)[0] == 0:
            return tf.constant(0.0, dtype=tf.float32)
        
        W_frozen = tf.stop_gradient(class_weights)  # CORRECTED
        W_norm = tf.nn.l2_normalize(W_frozen, axis=0)
        
        similarities = tf.matmul(unknown_embeddings, W_norm)
        max_similarities = tf.reduce_max(similarities, axis=1)
        loss = tf.reduce_mean(max_similarities)
        
        return loss

center_loss_layer = CenterLossLayer(train_data.num_classes, CFG.EMBED_DIM, alpha=0.5)
unknown_rejection_layer = UnknownRejectionLoss()

# ============================================================================
# PHYSICS LOSSES (SYMMETRY + SMOOTHNESS ONLY)
# ============================================================================
# Note: Energy loss removed due to geometric incompatibility with L2-normalized embeddings
# The energy gradient is necessarily tangent to the unit hypersphere, making
# directional alignment impossible (cos(∇E, z) = 0 by construction)

def get_physics_weights_tf(epoch, warmup_epochs=10):
    """TF-native weight scheduler"""
    epoch = tf.cast(epoch, tf.float32)
    warmup_epochs = tf.cast(warmup_epochs, tf.float32)
    scale = tf.minimum(epoch / warmup_epochs, 1.0)
    
    return (
        CFG.PHYSICS_SYM_WEIGHT * scale,
        0.0,  # Energy removed
        CFG.PHYSICS_LAP_WEIGHT * scale
    )

# ============================================================================
# TRAINER (with all corrections)
# ============================================================================

class PhysicsConstrainedTrainer(keras.Model):
    """CORRECTED trainer with physics losses only on known samples"""
    
    def __init__(self, base_model, embedding_model, center_loss_layer,
                 unknown_rejection_layer, center_loss_weight, unknown_loss_weight,
                 physics_warmup_epochs, **kwargs):
        super().__init__(**kwargs)
        
        self.base_model = base_model
        self.embedding_model = embedding_model
        self.center_loss_layer = center_loss_layer
        self.unknown_rejection_layer = unknown_rejection_layer
        
        self.center_loss_weight = center_loss_weight
        self.unknown_loss_weight = unknown_loss_weight
        self.physics_warmup_epochs = physics_warmup_epochs
        
        self.current_epoch = tf.Variable(0, trainable=False, dtype=tf.int32)
        
        # Metrics
        self.arcface_loss_tracker = keras.metrics.Mean(name="arcface_loss")
        self.center_loss_tracker = keras.metrics.Mean(name="center_loss")
        self.unknown_loss_tracker = keras.metrics.Mean(name="unknown_loss")
        self.sym_loss_tracker = keras.metrics.Mean(name="sym_loss")
        self.energy_loss_tracker = keras.metrics.Mean(name="energy_loss")
        self.lap_loss_tracker = keras.metrics.Mean(name="lap_loss")
        self.total_loss_tracker = keras.metrics.Mean(name="loss")
        self.accuracy_tracker = keras.metrics.SparseCategoricalAccuracy(name="accuracy")
        self.embed_norm_tracker = keras.metrics.Mean(name="embed_norm")
    
    def call(self, inputs):
        return self.base_model(inputs)
    
    def train_step(self, data):
        inputs, labels = data
        face_input = inputs['face_input']
        
        lambda_sym, lambda_energy, lambda_lap = get_physics_weights_tf(
            self.current_epoch,
            warmup_epochs=self.physics_warmup_epochs
        )
        
        # CRITICAL: Prepare augmented versions OUTSIDE gradient tape
        # (augmentation ops don't have gradients defined)
        face_flipped = tf.image.flip_left_right(face_input)
        # For smoothness loss, use simple operations with gradients
        # Instead of light_augmentation which uses ops without gradients
        
        with tf.GradientTape() as main_tape:
            logits, embeddings = self.base_model(inputs, training=True)
            
            labels = tf.cast(labels, tf.int32)
            known_mask = tf.greater_equal(labels, 0)
            known_embeddings = tf.boolean_mask(embeddings, known_mask)
            
            # ArcFace
            def compute_arcface():
                known_logits = tf.boolean_mask(logits, known_mask)
                known_labels = tf.boolean_mask(labels, known_mask)
                return tf.reduce_mean(
                    tf.nn.sparse_softmax_cross_entropy_with_logits(
                        labels=known_labels,
                        logits=known_logits
                    )
                )
            
            arcface_loss = tf.cond(
                tf.reduce_any(known_mask),
                compute_arcface,
                lambda: tf.constant(0.0, dtype=tf.float32)
            )
            
            # Center loss
            center_loss = self.center_loss_layer(embeddings, labels, training=True)
            
            # Unknown loss
            arcface_layer = self.base_model.get_layer('arcface')
            class_weights = arcface_layer.W
            unknown_loss = self.unknown_rejection_layer(embeddings, labels, class_weights)
            
            # Physics losses (KNOWN SAMPLES ONLY)
            def compute_physics_losses():
                known_face_input = tf.boolean_mask(face_input, known_mask)
                
                # Flip (no gradient needed, use stop_gradient)
                known_flipped = tf.stop_gradient(tf.image.flip_left_right(known_face_input))
                
                # For smoothness, use simple additive noise instead of contrast/brightness
                # This has gradients and won't cause issues
                noise = tf.random.normal(tf.shape(known_face_input), mean=0.0, stddev=0.1)
                known_augmented = known_face_input + noise
                known_augmented = tf.stop_gradient(known_augmented)
                
                embeddings_flip_known = self.embedding_model(known_flipped, training=True)
                embeddings_aug_known = self.embedding_model(known_augmented, training=True)
                
                # L_sym (Symmetry)
                diff_sym = known_embeddings - embeddings_flip_known
                sym = tf.reduce_mean(tf.reduce_sum(tf.square(diff_sym), axis=1))
                
                # L_lap (Smoothness)
                diff_lap = known_embeddings - embeddings_aug_known
                lap = tf.reduce_mean(tf.reduce_sum(tf.square(diff_lap), axis=1))
                
                return sym, lap
            
            sym_loss, lap_loss = tf.cond(
                tf.reduce_any(known_mask),
                compute_physics_losses,
                lambda: (tf.constant(0.0, dtype=tf.float32),
                        tf.constant(0.0, dtype=tf.float32))
            )
            
            # Energy loss set to zero (removed)
            energy_loss = tf.constant(0.0, dtype=tf.float32)
            
            # Total loss (WITHOUT ENERGY)
            # L = L_arc + λ1·L_center + λ2·L_unknown + λ3·L_sym + λ4·L_lap
            total_loss = (
                arcface_loss +
                self.center_loss_weight * center_loss +
                self.unknown_loss_weight * unknown_loss +
                lambda_sym * sym_loss +
                # lambda_energy * energy_loss +  # REMOVED
                lambda_lap * lap_loss
            )
            
            if self.base_model.losses:
                total_loss += tf.add_n(self.base_model.losses)
        
        gradients = main_tape.gradient(total_loss, self.base_model.trainable_variables)
        self.optimizer.apply_gradients(zip(gradients, self.base_model.trainable_variables))
        
        # Update metrics
        self.arcface_loss_tracker.update_state(arcface_loss)
        self.center_loss_tracker.update_state(center_loss)
        self.unknown_loss_tracker.update_state(unknown_loss)
        self.sym_loss_tracker.update_state(sym_loss)
        self.energy_loss_tracker.update_state(energy_loss)
        self.lap_loss_tracker.update_state(lap_loss)
        self.total_loss_tracker.update_state(total_loss)
        
        embed_norms = tf.norm(embeddings, axis=1)
        self.embed_norm_tracker.update_state(tf.reduce_mean(embed_norms))
        
        def update_acc():
            known_logits = tf.boolean_mask(logits, known_mask)
            known_labels = tf.boolean_mask(labels, known_mask)
            self.accuracy_tracker.update_state(known_labels, known_logits)
            return tf.constant(0)
        
        tf.cond(tf.reduce_any(known_mask), update_acc, lambda: tf.constant(0))
        
        return {
            "loss": self.total_loss_tracker.result(),
            "arcface_loss": self.arcface_loss_tracker.result(),
            "center_loss": self.center_loss_tracker.result(),
            "unknown_loss": self.unknown_loss_tracker.result(),
            "sym_loss": self.sym_loss_tracker.result(),
            "energy_loss": self.energy_loss_tracker.result(),
            "lap_loss": self.lap_loss_tracker.result(),
            "accuracy": self.accuracy_tracker.result(),
            "embed_norm": self.embed_norm_tracker.result()
        }
    
    @property
    def metrics(self):
        return [
            self.total_loss_tracker,
            self.arcface_loss_tracker,
            self.center_loss_tracker,
            self.unknown_loss_tracker,
            self.sym_loss_tracker,
            self.energy_loss_tracker,
            self.lap_loss_tracker,
            self.accuracy_tracker,
            self.embed_norm_tracker
        ]

trainer = PhysicsConstrainedTrainer(
    base_model=model,
    embedding_model=embedding_model,
    center_loss_layer=center_loss_layer,
    unknown_rejection_layer=unknown_rejection_layer,
    center_loss_weight=CFG.CENTER_LOSS_WEIGHT,
    unknown_loss_weight=CFG.UNKNOWN_LOSS_WEIGHT,
    physics_warmup_epochs=CFG.PHYSICS_WARMUP_EPOCHS
)

print("\n✓ Trainer initialized (all corrections applied)")

# ============================================================================
# CURRICULUM MIXING
# ============================================================================

def make_curriculum_dataset(known_ds, unknown_ds, unknown_ratio):
    """
    Mix known (face_train) with unknowns (face_val heavily augmented)
    
    CORRECTED: Actually uses unknown_ds (face_val), not slicing from known_ds
    """
    if unknown_ratio == 0.0:
        return known_ds.batch(CFG.BATCH_SIZE).prefetch(CFG.AUTOTUNE)
    
    n_known = max(4, int(CFG.BATCH_SIZE * (1.0 - unknown_ratio)))
    n_unknown = CFG.BATCH_SIZE - n_known
    
    print(f"  Batch: {n_known} known (face_train) + {n_unknown} unknown (face_val augmented)")
    
    # CORRECTED: Separate batching for known and unknown
    known_batched = known_ds.batch(n_known)
    unknown_batched = unknown_ds.batch(n_unknown)
    
    # Zip them together
    mixed_ds = tf.data.Dataset.zip((known_batched, unknown_batched))
    
    def create_mixed_batch(known_batch, unknown_batch):
        known_inputs, known_labels = known_batch
        unknown_inputs, unknown_labels = unknown_batch
        
        # Known faces from face_train
        known_faces = known_inputs['face_input']
        
        # Unknown faces from face_val (heavily augmented)
        unknown_faces_source = unknown_inputs['face_input']
        unknown_faces = tf.map_fn(augment_unknown, unknown_faces_source, dtype=tf.float32)
        unknown_labels_new = tf.fill([tf.shape(unknown_faces)[0]], -1)
        
        # Merge
        merged_faces = tf.concat([known_faces, unknown_faces], axis=0)
        merged_labels = tf.concat([known_labels, unknown_labels_new], axis=0)
        
        # Shuffle
        idx = tf.random.shuffle(tf.range(tf.shape(merged_labels)[0]))
        merged_faces = tf.gather(merged_faces, idx)
        merged_labels = tf.gather(merged_labels, idx)
        
        merged_inputs = {
            'face_input': merged_faces,
            'label_input': merged_labels
        }
        
        return merged_inputs, merged_labels
    
    return mixed_ds.map(create_mixed_batch, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

# ============================================================================
# CALLBACKS
# ============================================================================

class EpochCounterCallback(keras.callbacks.Callback):
    def __init__(self, trainer):
        super().__init__()
        self.trainer = trainer
    
    def on_epoch_begin(self, epoch, logs=None):
        self.trainer.current_epoch.assign(epoch)
        
        lambda_sym, lambda_energy, lambda_lap = get_physics_weights_tf(
            epoch,
            warmup_epochs=self.trainer.physics_warmup_epochs
        )
        print(f"\n  [Epoch {epoch+1}] Physics: "
              f"λ_sym={lambda_sym:.5f}, λ_lap={lambda_lap:.5f} (energy removed)")

# ============================================================================
# TRAINING
# ============================================================================

print("\n" + "=" * 70)
print("TRAINING")
print("=" * 70)

print(f"\nPhase 1 ({CFG.PHASE_1_EPOCHS} epochs): 100% known (face_train)")
print(f"Phase 2 ({CFG.PHASE_2_EPOCHS} epochs): 80% known, 20% unknown (face_val)")
print(f"Phase 3 ({CFG.PHASE_3_EPOCHS} epochs): 60% known, 40% unknown (face_val)")

# CRITICAL: Build model variables before training
# This is required for subclassed models
print("\n✓ Building model variables...")
train_ds_p1_temp = train_data.get_tfdata_unbatched(shuffle=True, augment=True)
dummy_batch = next(iter(train_ds_p1_temp.batch(CFG.BATCH_SIZE)))
_ = trainer(dummy_batch[0], training=False)
print("  Model variables created")

trainer.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3))

# PHASE 1: Only face_train
print("\n" + "=" * 70)
print(f"PHASE 1: Warmup ({CFG.PHASE_1_EPOCHS} epochs)")
print("=" * 70)

train_ds_p1 = train_data.get_tfdata_unbatched(shuffle=True, augment=True)
train_ds_p1 = make_curriculum_dataset(train_ds_p1, None, unknown_ratio=0.0)

phase1_callbacks = [
    EpochCounterCallback(trainer),
    keras.callbacks.ModelCheckpoint(
        'physics_final_phase1.weights.h5',
        monitor='loss',
        save_best_only=True,
        save_weights_only=True,
        verbose=1
    )
]

history_p1 = trainer.fit(
    train_ds_p1,
    steps_per_epoch=CFG.STEPS_PER_EPOCH,
    epochs=CFG.PHASE_1_EPOCHS,
    callbacks=phase1_callbacks
)

# PHASE 2: face_train + face_val (20% unknown)
print("\n" + "=" * 70)
print(f"PHASE 2: Introduction ({CFG.PHASE_2_EPOCHS} epochs)")
print("=" * 70)

train_ds_p2 = train_data.get_tfdata_unbatched(shuffle=True, augment=True)
unknown_ds_p2 = unknown_source_data.get_tfdata_unbatched(shuffle=True, augment=False)
train_ds_p2_mixed = make_curriculum_dataset(train_ds_p2, unknown_ds_p2, unknown_ratio=0.2)

# CRITICAL: Ensure variables exist before loading weights
# (Should already be built from Phase 1, but doing this for safety)
print("\n✓ Ensuring model variables are built...")
dummy_batch_p2 = next(iter(train_ds_p2_mixed))
_ = trainer(dummy_batch_p2[0], training=False)
print("  Variables confirmed")

trainer.load_weights('physics_final_phase1.weights.h5')
print("  Loaded Phase 1 weights")

phase2_callbacks = [
    EpochCounterCallback(trainer),
    keras.callbacks.ModelCheckpoint(
        'physics_final_phase2.weights.h5',
        monitor='loss',
        save_best_only=True,
        save_weights_only=True,
        verbose=1
    )
]

history_p2 = trainer.fit(
    train_ds_p2_mixed,
    steps_per_epoch=CFG.STEPS_PER_EPOCH,
    initial_epoch=CFG.PHASE_1_EPOCHS,
    epochs=CFG.PHASE_1_EPOCHS + CFG.PHASE_2_EPOCHS,
    callbacks=phase2_callbacks
)

# PHASE 3: face_train + face_val (40% unknown, lower LR)
print("\n" + "=" * 70)
print(f"PHASE 3: Hardening ({CFG.PHASE_3_EPOCHS} epochs)")
print("=" * 70)

train_ds_p3 = train_data.get_tfdata_unbatched(shuffle=True, augment=True)
unknown_ds_p3 = unknown_source_data.get_tfdata_unbatched(shuffle=True, augment=False)
train_ds_p3_mixed = make_curriculum_dataset(train_ds_p3, unknown_ds_p3, unknown_ratio=0.4)

# CRITICAL: Ensure variables exist before loading weights
print("\n✓ Ensuring model variables are built...")
dummy_batch_p3 = next(iter(train_ds_p3_mixed))
_ = trainer(dummy_batch_p3[0], training=False)
print("  Variables confirmed")

trainer.load_weights('physics_final_phase2.weights.h5')
print("  Loaded Phase 2 weights")

trainer.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-4))

phase3_callbacks = [
    EpochCounterCallback(trainer),
    keras.callbacks.ModelCheckpoint(
        'physics_final_best.weights.h5',
        monitor='loss',
        save_best_only=True,
        save_weights_only=True,
        verbose=1
    )
]

history_p3 = trainer.fit(
    train_ds_p3_mixed,
    steps_per_epoch=CFG.STEPS_PER_EPOCH,
    initial_epoch=CFG.PHASE_1_EPOCHS + CFG.PHASE_2_EPOCHS,
    epochs=CFG.TOTAL_EPOCHS,
    callbacks=phase3_callbacks
)

print("\n" + "=" * 70)
print("TRAINING COMPLETE")
print("=" * 70)

# ============================================================================
# EXPORT
# ============================================================================

print("\n✓ Exporting inference model...")

trainer.load_weights('physics_final_best.weights.h5')

inference_model = keras.Model(
    inputs=trainer.base_model.input[0],
    outputs=trainer.base_model.output[1],
    name="physics_final_inference"
)

tf.saved_model.save(inference_model, "physics_final_inference_model")
print("  Saved: physics_final_inference_model/")

arcface_layer = model.get_layer('arcface')
class_centers = arcface_layer.W.numpy()
class_centers_norm = class_centers / np.linalg.norm(class_centers, axis=0, keepdims=True)
np.save('physics_final_class_centers.npy', class_centers_norm.T)
print("  Saved: physics_final_class_centers.npy")

metadata = {
    'model_type': 'physics_constrained_final',
    'training_classes': train_data.classes,
    'num_training_classes': train_data.num_classes,
    'unknown_source': 'face_val',
    'test_set': 'face_test (untouched)',
    'corrections': [
        'UnknownRejectionLoss freezes class_weights',
        'CenterLoss updates only during training',
        'Physics losses applied only to known samples',
        'Proper data split: train on face_train, unknowns from face_val'
    ],
    'physics_config': {
        'sym_weight': CFG.PHYSICS_SYM_WEIGHT,
        'energy_weight': CFG.PHYSICS_ENERGY_WEIGHT,
        'lap_weight': CFG.PHYSICS_LAP_WEIGHT,
        'energy_tau': CFG.ENERGY_TAU,
        'warmup_epochs': CFG.PHYSICS_WARMUP_EPOCHS
    }
}

with open('physics_final_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)
print("  Saved: physics_final_metadata.json")

print("\n✓ All done! face_test remains untouched for final evaluation.")
print("=" * 70)

PHYSICS-CONSTRAINED OPEN-SET LIVESTOCK IDENTIFICATION
FINAL VERSION (Energy Loss Removed)

Final Loss Function:
  L = L_arc + λ1·L_center + λ2·L_unknown + λ3·L_sym + λ4·L_lap

Data Strategy:
  face_train: Training set (learn identities)
  face_val:   Unknown source (synthetic unknowns during training)
  face_test:  UNTOUCHED (final evaluation only)

Note: Energy loss removed due to geometric incompatibility

LOADING DATA

✓ Training set (face_train):
  345 cows, 3510 images
  Classes: ['DEONI-19-C2082', 'GB-5-C8012', 'GR-6-C2004', 'GRI-4-C25', 'GRI-5-C12']...

✓ Unknown source (face_val):
  30 cows, 291 images
  These cows will serve as synthetic unknowns during training
  Classes: ['HALLIKAR-7-C5027', 'HF-2-C1016', 'HF-2-C2087', 'HF-2-C5042', 'HF-2-C6004']...

✓ Test set (face_test): UNTOUCHED
  Will be used only for final open-set evaluation

BUILDING MODEL


2026-02-14 14:30:22.105875: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-14 14:30:22.783722: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1616] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 14648 MB memory:  -> device: 0, name: Quadro RTX 5000, pci bus id: 0000:d8:00.0, compute capability: 7.5



✓ Model built for 345 training classes
  Architecture: EfficientNetB0 → Dense(512) → L2-Normalize → ArcFace
✓ Embedding extraction model created

✓ Trainer initialized (all corrections applied)

TRAINING

Phase 1 (5 epochs): 100% known (face_train)
Phase 2 (15 epochs): 80% known, 20% unknown (face_val)
Phase 3 (10 epochs): 60% known, 40% unknown (face_val)

✓ Building model variables...


2026-02-14 14:30:28.498711: I tensorflow/stream_executor/cuda/cuda_dnn.cc:384] Loaded cuDNN version 8201


  Model variables created

PHASE 1: Warmup (5 epochs)

  [Epoch 1] Physics: λ_sym=0.00000, λ_lap=0.00000 (energy removed)
Epoch 1/5
150/150 [==============================] - ETA: 0s - loss: 24.8087 - arcface_loss: 24.6593 - center_loss: 0.8023 - unknown_loss: 0.0000e+00 - sym_loss: 0.7585 - energy_loss: 0.0000e+00 - lap_loss: 0.7000 - accuracy: 0.0446 - embed_norm: 1.0000
Epoch 1: loss improved from inf to 24.80870, saving model to physics_final_phase1.weights.h5
150/150 [==============================] - 54s 142ms/step - loss: 24.8087 - arcface_loss: 24.6593 - center_loss: 0.8023 - unknown_loss: 0.0000e+00 - sym_loss: 0.7585 - energy_loss: 0.0000e+00 - lap_loss: 0.7000 - accuracy: 0.0446 - embed_norm: 1.0000

  [Epoch 2] Physics: λ_sym=0.00100, λ_lap=0.00100 (energy removed)
Epoch 2/5
150/150 [==============================] - ETA: 0s - loss: 6.6341 - arcface_loss: 6.4717 - center_loss: 0.6599 - unknown_loss: 0.0000e+00 - sym_loss: 0.6496 - energy_loss: 0.0000e+00 - lap_loss: 0.5921 

INFO:tensorflow:Assets written to: physics_final_inference_model/assets


INFO:tensorflow:Assets written to: physics_final_inference_model/assets


  Saved: physics_final_inference_model/
  Saved: physics_final_class_centers.npy
  Saved: physics_final_metadata.json

✓ All done! face_test remains untouched for final evaluation.


In [3]:
import tensorflow as tf
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
import json

"""
COMPREHENSIVE UNKNOWN TEST SET EVALUATION

Generates detailed Excel report with:
1. Per-image analysis (every test image with its match)
2. Per-class analysis (aggregated statistics per cow)
3. Threshold sweep (FAR analysis across thresholds)
4. Gallery information (training classes)
5. Summary statistics
6. Visualizations

This script evaluates the physics-constrained model (WITHOUT energy loss)
against the completely unseen face_test dataset.
"""

np.random.seed(42)
tf.random.set_seed(42)

# ============================================================================
# CONFIGURATION
# ============================================================================

DATASET_ROOT = "/home/student/Desktop/Kumar/dataset"
MODEL_PATH = "physics_final_inference_model"
CENTERS_PATH = "physics_final_class_centers.npy"
METADATA_PATH = "physics_final_metadata.json"

class CFG:
    IMG_SIZE = 224

print("=" * 80)
print("COMPREHENSIVE UNKNOWN TEST SET EVALUATION")
print("=" * 80)
print(f"\nDataset: {DATASET_ROOT}")
print(f"Model:   {MODEL_PATH}")

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def load_image(path):
    """Load and preprocess image"""
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, (CFG.IMG_SIZE, CFG.IMG_SIZE))
    img = tf.cast(img, tf.float32)
    img = tf.keras.applications.efficientnet.preprocess_input(img)
    return img

class DatasetLoader:
    """Load cattle dataset from folder"""
    def __init__(self, folder_path):
        self.cattle_data = {}
        
        if not os.path.exists(folder_path):
            raise ValueError(f"Folder not found: {folder_path}")
        
        for cow_id in sorted(os.listdir(folder_path)):
            cow_path = os.path.join(folder_path, cow_id)
            if not os.path.isdir(cow_path):
                continue
            
            imgs = [os.path.join(cow_path, f) for f in os.listdir(cow_path)
                   if f.lower().endswith((".jpg", ".jpeg", ".png"))]
            
            if len(imgs) > 0:
                self.cattle_data[cow_id] = imgs
        
        self.classes = sorted(list(self.cattle_data.keys()))
        self.num_classes = len(self.classes)
        self.total_images = sum(len(v) for v in self.cattle_data.values())

# ============================================================================
# LOAD DATA
# ============================================================================

print("\n" + "=" * 80)
print("LOADING DATASETS")
print("=" * 80)

face_train_path = os.path.join(DATASET_ROOT, "face_train")
face_test_path = os.path.join(DATASET_ROOT, "face_test")

train_loader = DatasetLoader(face_train_path)
test_loader = DatasetLoader(face_test_path)

print(f"\n✓ Gallery (face_train): {train_loader.num_classes} cattle, {train_loader.total_images} images")
print(f"✓ Unknown test (face_test): {test_loader.num_classes} cattle, {test_loader.total_images} images")

# ============================================================================
# LOAD MODEL
# ============================================================================

print("\n" + "=" * 80)
print("LOADING MODEL")
print("=" * 80)

loaded_model = tf.saved_model.load(MODEL_PATH)
infer_fn = loaded_model.signatures["serving_default"]
input_name = list(infer_fn.structured_input_signature[1].keys())[0]
output_name = list(infer_fn.structured_outputs.keys())[0]
print(f"✓ Model loaded")

class_centers = np.load(CENTERS_PATH)
print(f"✓ Class centers loaded: {class_centers.shape[0]} classes")

with open(METADATA_PATH, 'r') as f:
    metadata = json.load(f)
training_classes = metadata['training_classes']
print(f"✓ Metadata loaded: {len(training_classes)} training classes")

def get_embedding(img_path):
    """Extract embedding from image"""
    try:
        img = load_image(img_path)[None]
        output = infer_fn(**{input_name: img})
        emb = output[output_name].numpy()[0]
        emb = emb / (np.linalg.norm(emb) + 1e-8)
        return emb
    except Exception as e:
        print(f"Error processing {img_path}: {e}")
        return None

# ============================================================================
# BUILD GALLERY
# ============================================================================

print("\n" + "=" * 80)
print("BUILDING GALLERY")
print("=" * 80)

print(f"Processing {train_loader.num_classes} training cattle...")

gallery = {}
gallery_details = []

for cow_id, img_list in train_loader.cattle_data.items():
    embeddings = []
    for img_path in img_list:
        emb = get_embedding(img_path)
        if emb is not None:
            embeddings.append(emb)
    
    if len(embeddings) > 0:
        avg_emb = np.mean(embeddings, axis=0)
        avg_emb = avg_emb / (np.linalg.norm(avg_emb) + 1e-8)
        gallery[cow_id] = avg_emb
        
        gallery_details.append({
            'Cattle_ID': cow_id,
            'Num_Images': len(img_list),
            'Enrolled_Images': len(embeddings),
            'Failed_Images': len(img_list) - len(embeddings)
        })

print(f"✓ Gallery built: {len(gallery)} cattle enrolled")
print(f"  Total images: {sum([d['Enrolled_Images'] for d in gallery_details])}")

gallery_df = pd.DataFrame(gallery_details)

# ============================================================================
# PROCESS UNKNOWN TEST SET
# ============================================================================

print("\n" + "=" * 80)
print("PROCESSING UNKNOWN TEST SET (face_test)")
print("=" * 80)

print(f"Processing {test_loader.num_classes} unknown cattle...")

per_image_results = []
failed_images = []

processed_count = 0
total_test_images = test_loader.total_images

for cow_id, img_list in test_loader.cattle_data.items():
    for img_idx, img_path in enumerate(img_list):
        processed_count += 1
        
        if processed_count % 100 == 0:
            print(f"  Progress: {processed_count}/{total_test_images} images ({processed_count/total_test_images*100:.1f}%)")
        
        emb = get_embedding(img_path)
        
        if emb is None:
            failed_images.append({
                'Test_Cattle_ID': cow_id,
                'Image_Path': img_path,
                'Image_Name': os.path.basename(img_path),
                'Error': 'Failed to extract embedding'
            })
            continue
        
        # Compute similarity to all gallery cattle
        similarities = {}
        for gallery_cow_id, gallery_emb in gallery.items():
            sim = np.dot(emb, gallery_emb)
            similarities[gallery_cow_id] = sim
        
        # Get top matches
        sorted_matches = sorted(similarities.items(), key=lambda x: x[1], reverse=True)
        
        top1_id, top1_score = sorted_matches[0]
        top2_id, top2_score = sorted_matches[1] if len(sorted_matches) > 1 else ('', 0.0)
        top3_id, top3_score = sorted_matches[2] if len(sorted_matches) > 2 else ('', 0.0)
        
        per_image_results.append({
            'Test_Cattle_ID': cow_id,
            'Image_Path': img_path,
            'Image_Name': os.path.basename(img_path),
            'Image_Number_In_Class': img_idx + 1,
            'Top1_Match_ID': top1_id,
            'Top1_Similarity': top1_score,
            'Top2_Match_ID': top2_id,
            'Top2_Similarity': top2_score,
            'Top3_Match_ID': top3_id,
            'Top3_Similarity': top3_score,
            'Embedding_Norm': np.linalg.norm(emb)
        })

print(f"\n✓ Processed {len(per_image_results)} images successfully")
if len(failed_images) > 0:
    print(f"  ⚠ Failed to process {len(failed_images)} images")

per_image_df = pd.DataFrame(per_image_results)
failed_df = pd.DataFrame(failed_images) if len(failed_images) > 0 else pd.DataFrame()

# ============================================================================
# PER-CLASS ANALYSIS
# ============================================================================

print("\n" + "=" * 80)
print("PER-CLASS ANALYSIS")
print("=" * 80)

per_class_stats = []

for cow_id in test_loader.cattle_data.keys():
    cow_images = per_image_df[per_image_df['Test_Cattle_ID'] == cow_id]
    
    if len(cow_images) == 0:
        continue
    
    per_class_stats.append({
        'Test_Cattle_ID': cow_id,
        'Total_Images': len(cow_images),
        'Avg_Top1_Similarity': cow_images['Top1_Similarity'].mean(),
        'Max_Top1_Similarity': cow_images['Top1_Similarity'].max(),
        'Min_Top1_Similarity': cow_images['Top1_Similarity'].min(),
        'Std_Top1_Similarity': cow_images['Top1_Similarity'].std(),
        'Most_Common_Match': cow_images['Top1_Match_ID'].mode()[0] if len(cow_images) > 0 else '',
        'Unique_Top1_Matches': cow_images['Top1_Match_ID'].nunique()
    })

per_class_df = pd.DataFrame(per_class_stats)

print(f"✓ Per-class statistics computed for {len(per_class_df)} cattle")

# ============================================================================
# THRESHOLD SWEEP
# ============================================================================

print("\n" + "=" * 80)
print("THRESHOLD SWEEP ANALYSIS")
print("=" * 80)

thresholds = np.arange(0.50, 0.91, 0.01)
threshold_results = []

print(f"Testing {len(thresholds)} thresholds from {thresholds[0]:.2f} to {thresholds[-1]:.2f}...")

for tau in thresholds:
    # Count false accepts (images with similarity >= threshold)
    false_accepts = (per_image_df['Top1_Similarity'] >= tau).sum()
    rejected = (per_image_df['Top1_Similarity'] < tau).sum()
    
    far = (false_accepts / len(per_image_df)) * 100
    reject_rate = (rejected / len(per_image_df)) * 100
    
    # Per-class false accept rate
    class_far_list = []
    for cow_id in test_loader.cattle_data.keys():
        cow_images = per_image_df[per_image_df['Test_Cattle_ID'] == cow_id]
        if len(cow_images) > 0:
            cow_fa = (cow_images['Top1_Similarity'] >= tau).sum()
            cow_far = (cow_fa / len(cow_images)) * 100
            class_far_list.append(cow_far)
    
    # Count cattle with 0% and 100% FAR at this threshold
    cattle_0_far = sum([1 for far in class_far_list if far == 0])
    cattle_100_far = sum([1 for far in class_far_list if far == 100])
    
    threshold_results.append({
        'Threshold': tau,
        'False_Accepts': false_accepts,
        'Rejected': rejected,
        'FAR_%': far,
        'Rejection_Rate_%': reject_rate,
        'Cattle_0%_FAR': cattle_0_far,
        'Cattle_100%_FAR': cattle_100_far,
        'Avg_Class_FAR_%': np.mean(class_far_list) if class_far_list else 0,
        'Std_Class_FAR_%': np.std(class_far_list) if class_far_list else 0
    })

threshold_df = pd.DataFrame(threshold_results)

print(f"✓ Threshold sweep complete")

# Find optimal thresholds
best_far_idx = threshold_df['FAR_%'].idxmin()
best_far_row = threshold_df.iloc[best_far_idx]

# Find thresholds meeting specific FAR targets
far_targets = [1.0, 2.0, 5.0, 10.0, 20.0]
target_thresholds = {}

for target in far_targets:
    matching = threshold_df[threshold_df['FAR_%'] <= target]
    if len(matching) > 0:
        target_thresholds[target] = matching.iloc[0]
    else:
        target_thresholds[target] = None

print(f"\nKey Operating Points:")
print(f"  Minimum FAR: {best_far_row['FAR_%']:.2f}% @ threshold {best_far_row['Threshold']:.2f}")

for target, row in target_thresholds.items():
    if row is not None:
        print(f"  FAR ≤ {target:.0f}%: {row['FAR_%']:.2f}% @ threshold {row['Threshold']:.2f}")
    else:
        print(f"  FAR ≤ {target:.0f}%: Not achievable")

# ============================================================================
# DETAILED ANALYSIS AT KEY THRESHOLDS
# ============================================================================

print("\n" + "=" * 80)
print("DETAILED ANALYSIS AT KEY THRESHOLDS")
print("=" * 80)

# Analyze at optimal threshold
optimal_threshold = best_far_row['Threshold']

per_image_df['Decision_Optimal'] = per_image_df['Top1_Similarity'].apply(
    lambda x: 'ACCEPTED' if x >= optimal_threshold else 'REJECTED'
)

# Per-class analysis at optimal threshold
per_class_at_optimal = []

for cow_id in test_loader.cattle_data.keys():
    cow_images = per_image_df[per_image_df['Test_Cattle_ID'] == cow_id]
    
    if len(cow_images) == 0:
        continue
    
    accepted = (cow_images['Top1_Similarity'] >= optimal_threshold).sum()
    rejected = (cow_images['Top1_Similarity'] < optimal_threshold).sum()
    
    per_class_at_optimal.append({
        'Test_Cattle_ID': cow_id,
        'Total_Images': len(cow_images),
        'False_Accepts': accepted,
        'Correct_Rejects': rejected,
        'FAR_%': (accepted / len(cow_images)) * 100,
        'Most_Similar_To': cow_images['Top1_Match_ID'].mode()[0] if len(cow_images) > 0 else '',
        'Avg_Similarity': cow_images['Top1_Similarity'].mean(),
        'Max_Similarity': cow_images['Top1_Similarity'].max()
    })

per_class_at_optimal_df = pd.DataFrame(per_class_at_optimal)

print(f"✓ Detailed analysis at optimal threshold ({optimal_threshold:.2f}) complete")

# ============================================================================
# VISUALIZATION
# ============================================================================

print("\n" + "=" * 80)
print("GENERATING VISUALIZATIONS")
print("=" * 80)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: FAR vs Threshold
ax1 = axes[0, 0]
ax1.plot(threshold_df['Threshold'], threshold_df['FAR_%'], 'b-', linewidth=2)
ax1.axhline(y=1, color='green', linestyle='--', alpha=0.5, label='1% FAR')
ax1.axhline(y=5, color='orange', linestyle='--', alpha=0.5, label='5% FAR')
ax1.axhline(y=10, color='red', linestyle='--', alpha=0.5, label='10% FAR')
ax1.axvline(x=optimal_threshold, color='black', linestyle=':', linewidth=2,
           label=f'Optimal ({optimal_threshold:.2f})')
ax1.set_xlabel('Similarity Threshold')
ax1.set_ylabel('False Accept Rate (%)')
ax1.set_title('FAR vs Threshold (Unknown Test Set)')
ax1.grid(True, alpha=0.3)
ax1.legend()

# Plot 2: Similarity Distribution
ax2 = axes[0, 1]
ax2.hist(per_image_df['Top1_Similarity'], bins=50, color='blue', alpha=0.7, edgecolor='black')
ax2.axvline(x=optimal_threshold, color='red', linestyle='--', linewidth=2,
           label=f'Optimal threshold ({optimal_threshold:.2f})')
ax2.set_xlabel('Top-1 Similarity Score')
ax2.set_ylabel('Frequency (Number of Images)')
ax2.set_title('Distribution of Similarity Scores')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Plot 3: Per-Class FAR Distribution at Optimal Threshold
ax3 = axes[1, 0]
ax3.hist(per_class_at_optimal_df['FAR_%'], bins=20, color='red', alpha=0.7, edgecolor='black')
ax3.set_xlabel('Per-Class FAR (%)')
ax3.set_ylabel('Number of Cattle')
ax3.set_title(f'Per-Class FAR Distribution @ Threshold {optimal_threshold:.2f}')
ax3.grid(True, alpha=0.3)

# Plot 4: Cattle with Perfect Rejection vs Full Accept
ax4 = axes[1, 1]
categories = ['0% FAR\n(Perfect)', '0-50% FAR', '50-100% FAR', '100% FAR\n(All Accepted)']
counts = [
    len(per_class_at_optimal_df[per_class_at_optimal_df['FAR_%'] == 0]),
    len(per_class_at_optimal_df[(per_class_at_optimal_df['FAR_%'] > 0) & 
                                (per_class_at_optimal_df['FAR_%'] <= 50)]),
    len(per_class_at_optimal_df[(per_class_at_optimal_df['FAR_%'] > 50) & 
                                (per_class_at_optimal_df['FAR_%'] < 100)]),
    len(per_class_at_optimal_df[per_class_at_optimal_df['FAR_%'] == 100])
]
colors = ['green', 'lightgreen', 'orange', 'red']
ax4.bar(categories, counts, color=colors, edgecolor='black')
ax4.set_ylabel('Number of Cattle')
ax4.set_title(f'Cattle Distribution by FAR @ Threshold {optimal_threshold:.2f}')
ax4.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('unknown_test_comprehensive_analysis.png', dpi=300, bbox_inches='tight')
print("✓ Saved: unknown_test_comprehensive_analysis.png")
plt.close()

# ============================================================================
# SAVE COMPREHENSIVE EXCEL REPORT
# ============================================================================

print("\n" + "=" * 80)
print("SAVING COMPREHENSIVE EXCEL REPORT")
print("=" * 80)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
excel_name = f"unknown_test_comprehensive_{timestamp}.xlsx"

print(f"Writing Excel file: {excel_name}")
print("  This may take a moment for large datasets...")

with pd.ExcelWriter(excel_name, engine='openpyxl') as writer:
    # Sheet 1: Summary
    summary_data = {
        'Metric': [
            'Model',
            'Training Cattle (Gallery)',
            'Training Images',
            'Unknown Test Cattle',
            'Unknown Test Images',
            'Failed Images',
            '',
            'OPTIMAL THRESHOLD:',
            'Threshold Value',
            'Minimum FAR',
            'Rejection Rate',
            'False Accepts',
            'Correct Rejects',
            '',
            'CATTLE BREAKDOWN @ OPTIMAL:',
            'Cattle with 0% FAR (Perfect)',
            'Cattle with 100% FAR (All Accepted)',
            'Cattle with 0-50% FAR',
            'Cattle with 50-100% FAR',
            '',
            'SIMILARITY STATISTICS:',
            'Mean Similarity',
            'Median Similarity',
            'Std Similarity',
            'Min Similarity',
            'Max Similarity'
        ],
        'Value': [
            'Physics-Constrained (Sym + Smooth)',
            str(train_loader.num_classes),
            str(train_loader.total_images),
            str(test_loader.num_classes),
            str(len(per_image_df)),
            str(len(failed_images)),
            '',
            '',
            f"{best_far_row['Threshold']:.2f}",
            f"{best_far_row['FAR_%']:.2f}%",
            f"{best_far_row['Rejection_Rate_%']:.2f}%",
            f"{best_far_row['False_Accepts']:.0f} / {len(per_image_df)}",
            f"{best_far_row['Rejected']:.0f} / {len(per_image_df)}",
            '',
            '',
            f"{counts[0]} ({counts[0]/test_loader.num_classes*100:.1f}%)",
            f"{counts[3]} ({counts[3]/test_loader.num_classes*100:.1f}%)",
            f"{counts[1]} ({counts[1]/test_loader.num_classes*100:.1f}%)",
            f"{counts[2]} ({counts[2]/test_loader.num_classes*100:.1f}%)",
            '',
            '',
            f"{per_image_df['Top1_Similarity'].mean():.4f}",
            f"{per_image_df['Top1_Similarity'].median():.4f}",
            f"{per_image_df['Top1_Similarity'].std():.4f}",
            f"{per_image_df['Top1_Similarity'].min():.4f}",
            f"{per_image_df['Top1_Similarity'].max():.4f}"
        ]
    }
    pd.DataFrame(summary_data).to_excel(writer, sheet_name='Summary', index=False)
    
    # Sheet 2: Per-Image Results (MOST DETAILED)
    per_image_df.to_excel(writer, sheet_name='Per_Image_Results', index=False)
    
    # Sheet 3: Per-Class Statistics
    per_class_df.to_excel(writer, sheet_name='Per_Class_Stats', index=False)
    
    # Sheet 4: Per-Class @ Optimal Threshold
    per_class_at_optimal_df.to_excel(writer, sheet_name='Per_Class_Optimal_Threshold', index=False)
    
    # Sheet 5: Threshold Sweep
    threshold_df.to_excel(writer, sheet_name='Threshold_Sweep', index=False)
    
    # Sheet 6: Gallery Info
    gallery_df.to_excel(writer, sheet_name='Gallery_Info', index=False)
    
    # Sheet 7: Failed Images (if any)
    if len(failed_df) > 0:
        failed_df.to_excel(writer, sheet_name='Failed_Images', index=False)

print(f"✓ Saved: {excel_name}")

# ============================================================================
# PRINT DETAILED SUMMARY
# ============================================================================

print("\n" + "=" * 80)
print("EVALUATION SUMMARY")
print("=" * 80)

print(f"\nDataset:")
print(f"  Gallery: {train_loader.num_classes} cattle ({train_loader.total_images} images)")
print(f"  Unknown Test: {test_loader.num_classes} cattle ({len(per_image_df)} images)")

print(f"\nOptimal Operating Point:")
print(f"  Threshold: {best_far_row['Threshold']:.2f}")
print(f"  FAR: {best_far_row['FAR_%']:.2f}%")
print(f"  Rejection Rate: {best_far_row['Rejection_Rate_%']:.2f}%")
print(f"  False Accepts: {best_far_row['False_Accepts']:.0f} / {len(per_image_df)}")

print(f"\nPer-Class Analysis @ Optimal Threshold:")
print(f"  Perfect rejection (0% FAR): {counts[0]} / {test_loader.num_classes} ({counts[0]/test_loader.num_classes*100:.1f}%)")
print(f"  All accepted (100% FAR): {counts[3]} / {test_loader.num_classes} ({counts[3]/test_loader.num_classes*100:.1f}%)")

print(f"\nSimilarity Statistics:")
print(f"  Mean: {per_image_df['Top1_Similarity'].mean():.4f}")
print(f"  Median: {per_image_df['Top1_Similarity'].median():.4f}")
print(f"  Std: {per_image_df['Top1_Similarity'].std():.4f}")
print(f"  Range: [{per_image_df['Top1_Similarity'].min():.4f}, {per_image_df['Top1_Similarity'].max():.4f}]")

print(f"\nTop 5 Most Confused Cattle (Highest Avg Similarity):")
top_confused = per_class_df.nlargest(5, 'Avg_Top1_Similarity')
for idx, row in top_confused.iterrows():
    print(f"  {row['Test_Cattle_ID']}: Avg={row['Avg_Top1_Similarity']:.4f}, "
          f"Max={row['Max_Top1_Similarity']:.4f}, Most similar to {row['Most_Common_Match']}")

print(f"\nTop 5 Most Distinct Cattle (Lowest Avg Similarity):")
top_distinct = per_class_df.nsmallest(5, 'Avg_Top1_Similarity')
for idx, row in top_distinct.iterrows():
    print(f"  {row['Test_Cattle_ID']}: Avg={row['Avg_Top1_Similarity']:.4f}, "
          f"Max={row['Max_Top1_Similarity']:.4f}")

print(f"\nTarget FAR Thresholds:")
for target, row in target_thresholds.items():
    if row is not None:
        print(f"  FAR ≤ {target:.0f}%: Threshold {row['Threshold']:.2f} "
              f"(Actual FAR: {row['FAR_%']:.2f}%, Rejection: {row['Rejection_Rate_%']:.1f}%)")
    else:
        print(f"  FAR ≤ {target:.0f}%: Not achievable")

print("\n" + "=" * 80)
print("EVALUATION COMPLETE")
print("=" * 80)

print(f"\n✓ Files generated:")
print(f"  - {excel_name}")
print(f"  - unknown_test_comprehensive_analysis.png")

print(f"\n✓ Excel sheets:")
print(f"  1. Summary - Overall statistics")
print(f"  2. Per_Image_Results - Every test image with matches")
print(f"  3. Per_Class_Stats - Aggregated per cattle")
print(f"  4. Per_Class_Optimal_Threshold - FAR per cattle at optimal threshold")
print(f"  5. Threshold_Sweep - FAR across all thresholds")
print(f"  6. Gallery_Info - Training cattle details")
if len(failed_df) > 0:
    print(f"  7. Failed_Images - Images that failed processing")

print("=" * 80)

COMPREHENSIVE UNKNOWN TEST SET EVALUATION

Dataset: /home/student/Desktop/Kumar/dataset
Model:   physics_final_inference_model

LOADING DATASETS

✓ Gallery (face_train): 345 cattle, 3510 images
✓ Unknown test (face_test): 93 cattle, 925 images

LOADING MODEL
✓ Model loaded
✓ Class centers loaded: 345 classes
✓ Metadata loaded: 345 training classes

BUILDING GALLERY
Processing 345 training cattle...
✓ Gallery built: 345 cattle enrolled
  Total images: 3510

PROCESSING UNKNOWN TEST SET (face_test)
Processing 93 unknown cattle...
  Progress: 100/925 images (10.8%)
  Progress: 200/925 images (21.6%)
  Progress: 300/925 images (32.4%)
  Progress: 400/925 images (43.2%)
  Progress: 500/925 images (54.1%)
  Progress: 600/925 images (64.9%)
  Progress: 700/925 images (75.7%)
  Progress: 800/925 images (86.5%)
  Progress: 900/925 images (97.3%)

✓ Processed 925 images successfully

PER-CLASS ANALYSIS
✓ Per-class statistics computed for 93 cattle

THRESHOLD SWEEP ANALYSIS
Testing 41 thresholds f